# Numerai Classic v6.0 ("Unity") — Deterministic Baselines: Bin-Snapping, Lag-Reversal & Persistence (`NumeraiExtractions` + `BinSnappingEstimator`, `LagReversalEstimator` & `PersistenceEstimator`)

- **Data Loading (`data_utils.load_unity_numerai_data`)**: Resolves and downloads the Unity v6.0 Parquets using the official Kaggle clients (`kagglehub.dataset_download` and `kaggle.api.kaggle_api_extended.KaggleApi`) with `numerapi.NumerAPI` fallback, catching each library's actual exception classes (`KaggleApiHTTPError`, `UnauthenticatedError`, `CredentialError`, `NotFoundError`, `DataCorruptionError`, `BackendError`, `KaggleEnvironmentError`, `ColabEnvironmentError`, `kaggle.rest.ApiException`, `HTTPError`, `OSError`, `ValueError`).
- **Exploratory Data Analysis (`# EDA`)**: Lists available `TARGET_COL` and `feature_set` options from `features.json`, inspects `train_df.head(2)` / `validation_df.head(2)` and `info()`, plots era sizes over time (`groupby(ERA_COL)[TARGET_COL].size()`), overlays feature standard-deviation distributions across Train vs. Validation, and displays the top 10 features correlated with `TARGET_COL`.

---

### 1. Feature Extraction Parameters (`utils.NumeraiExtractions` — `estimator__extractor__*`)
- **`era_col`** (`[ERA_COL]`): Column name containing the sequential weekly time index (`'era'`).
- **`ticker_col`** (`[TICKER_COL]`): Column name containing the security identifier (`'numerai_ticker'`).
- **`target_col`** (`[TARGET_COL, 'target_cyrusd_20', 'target_teager2b_20']`): Target column used when ranking features by target correlation.
- **`year_horizon_filter`** (`[None, (1, 2, 3), (4, 5, 6), (7, 8, 9)]`): Restricts `fit()` variance and correlation estimation to specific 52-week training year buckets `(era - 1) // 52 + 1` (`None` uses all training years; never exposed as a model feature).
- **`variance_threshold`** (`[None, 0.02, 0.05, 0.10]`): Drops near-constant features whose training variance is $\le$ threshold via `sklearn.feature_selection.VarianceThreshold` (`None` keeps all features).
- **`correlation_method`** (`[None, 'pandas_mean', 'numerai_mean', 'pearson', 'spearman']`):
  - `None`: Skips correlation ranking and keeps original column order.
  - `'pandas_mean'`: Computes per-era Pearson correlation (`era_df.corrwith(era_target)`) and averages across eras.
  - `'numerai_mean'`: Computes official per-era `numerai_tools.scoring.numerai_corr` and averages across eras.
  - `'pearson'`: Global pooled Pearson correlation (`r_regression`) across all rows.
  - `'spearman'`: Global pooled Spearman rank correlation across all rows.
- **`abs_corr`** (`[True, False]`): `True` ranks features by $|\text{corr}|$ (strongest magnitude in either direction); `False` ranks by signed $\text{corr}$ (positive only first).
- **`top_k`** (`[None, 10, 20, 40]`): Keeps only the top $K$ highest-ranked features from the extractor (`None` keeps all features).
- **`feature_selection_percentile`** (`[None, 25.0, 50.0, 75.0, 100.0]`): Keeps the top percentage of ranked features (`None` keeps all features).
- **`sparse_selector`** (`[None, 'lasso', 'elasticnet']`) & **`sparse_alpha`** (`[0.001, 0.01, 0.1]` / `Real(1e-4, 1e-1, prior='log-uniform')`): Optional multivariate sparse feature selection via `sklearn.feature_selection.SelectFromModel` with `Lasso` or `ElasticNet` fitted strictly inside `fit()` (`None` disables).
- **`pca_n_components`** (`[None, 10, 20, 40]` / `0.90`): Optional `sklearn.decomposition.PCA` compression fitted strictly inside `fit()` and quantized to `0..4` `int8` bins in `transform()` to reduce feature exposure without dropping features (`None` disables).
- **`quantize_scope`** (`['era', 'train_fit', 'global']`) & **`quantize_strategy`** (`['zscore', 'quantile']`): Controls how continuous derived features are quantized into `0..4` `int8` bins (`'era'` = leak-free within-era cross-sectional quantization; `'train_fit'` = frozen `StandardScaler` fitted strictly on training data in `fit()`; `'global'` = legacy full-array quantization; `'zscore'` = z-score rounding vs. `'quantile'` = `KBinsDiscretizer` quantile edges).
- **`sequence_top_n`** (`[4, 2, 8]`): Number of top-ranked features expanded by rolling-era, lag-diff, and per-ticker sequence extractors.
- **`missing_fill_value`** (`[2.0, 0.0, 4.0, 'mean', 'median', 'era_mean', 'era_median']`):
  - `2.0`, `0.0`, `4.0`, `'mean'`, `'median'`: Imputes missing values via `sklearn.impute.SimpleImputer` (`2.0` is the neutral middle bin of Numerai's `0..4` scale).
  - `'era_mean'` / `'era_median'`: Fills missing values with each feature's within-era cross-sectional mean or median.
- **`feature_groups`** (`unity_feature_groups` on `NumeraiExtractions`, `None` if no thematic groups exist in `feature_metadata['feature_sets']`; omitted from CV search spaces to prevent console log bloat) & **`basic_group_stats`** (`[None, 'mean', 'std', 'skew', 'mean|std', 'mean|skew', 'std|skew', 'mean|std|skew']`): Computes leak-free row-wise summary statistics (`mean`, `std`, `skew` quantized to `0..4` bins) across real Unity v6.0 feature groups or built-in thematic prefixes (`None` disables).
- **`fourier_period_eras`** (`[52, 26, 13]`): Cycle length in eras (52 = annual, 26 = semi-annual, 13 = quarterly) used by `fourier_order`.
- **`fourier_order`** (`[None, 1, 2, 3]`): Sole cyclical seasonality encoding — adds smooth $\sin(2\pi k \cdot \text{era} / P)$ and $\cos(2\pi k \cdot \text{era} / P)$ harmonics for $k=1..\text{fourier\_order}$ quantized to `0..4` bins (`None` disables).
- **`include_rolling_era`** (`[False, True]`) & **`rolling_era_windows`** (`[(4, 12), (2, 6, 12), (4, 8, 16)]`): When `include_rolling_era=True`, computes causal rolling era means over each window length $W$ in `rolling_era_windows` for the top correlated features, quantized to `0..4` bins.
- **`include_lag_diff`** (`[False, True]`) & **`era_lag_scales`** (`[(1, 2, 4), (1, 2, 4, 8), (2, 4, 8)]`): When `include_lag_diff=True`, computes causal era-to-era lag differences ($\bar{x}_e - \bar{x}_{e-k}$) for each lag step $k$ in `era_lag_scales`, quantized to `0..4` bins.
- **`ticker_features`** (`[None, 'rolling_mean', 'momentum_diff', 'weeks_since_high', 'rolling_rank', 'weeks_since_change', 'expanding_stats', 'raw_lags', 'rolling_mean|momentum_diff', 'rolling_mean|momentum_diff|rolling_rank', 'expanding_stats|raw_lags', 'rolling_mean|momentum_diff|weeks_since_high|rolling_rank|weeks_since_change|expanding_stats|raw_lags']`), **`ticker_windows`** (`[(4, 12), (2, 6, 12)]`), & **`ticker_lag_steps`** (`[None, (1, 2), (1, 2, 4)]`): Computes causal per-ticker time-series features (`rolling_mean`, `momentum_diff`, vectorized `weeks_since_high`, `rolling_rank`, vectorized `weeks_since_change`, `expanding_stats` [`expanding().mean()/min()/max()`], and `raw_lags` [`shift(k)` for $k \in$ `ticker_lag_steps`]) grouped strictly by `numerai_ticker` sorted chronologically by `era` (`None` disables).
- **`era_dispersion_features`** (`[None, 'mean', 'std', 'extreme_frac', 'mean|std', 'mean|extreme_frac', 'std|extreme_frac', 'mean|std|extreme_frac']`): Adds per-row cross-sectional mean (`'mean'`), standard deviation (`'std'`), and fraction of extreme `0`/`4` bins (`'extreme_frac'`) across selected features (`None` disables).

---

### 2. Shared Parent Estimator Parameters (`baseline_utils.BaseNumeraiEstimator` — `estimator__*`)
- **`correlation_method`** (`[None, 'pandas_mean', 'numerai_mean', 'pearson', 'spearman']`): How the estimator scores and weights post-extraction features against `TARGET_COL`.
- **`abs_corr`** (`[True, False]`): `True` selects top features by $|\text{corr}|$; `False` selects by signed $\text{corr}$.
- **`top_k`** (`[None, 10, 20, 40]`): Number of top post-extraction features used by the estimator (`None` uses all extracted features).
- **`era_weighting`** (`[None, 'equal_era', 'recency_linear', 'recency_exp']`):
  - `None`: Equal weight per row.
  - `'equal_era'`: Equal total weight per era ($1 / N_{\text{era}}$) so large eras do not dominate.
  - `'recency_linear'`: Linearly increases era weight from oldest ($0.5$) to newest ($1.5$).
  - `'recency_exp'`: Exponentially decays older eras by $\exp(-\text{recency\_decay} \cdot (e_{\max} - e))$.
- **`recency_decay`** (`Real(0.001, 0.020)`): Exponential decay rate per era when `era_weighting='recency_exp'`.
- **`prediction_transform`** (`[None, 'era_zscore', 'era_rank']`):
  - `None`: Returns raw model predictions.
  - `'era_zscore'`: Standardizes predictions to zero mean and unit variance within each era.
  - `'era_rank'`: Converts predictions to `[0, 1]` percentile ranks within each era.
- **`bias_correction`** (`[None, 'mean_residual']`): When `'mean_residual'`, records `mean(y_train - y_hat_train)` in `fit()` and adds it to predictions in `predict()` (`None` disables).

---

### 3. Child Baseline Estimator Parameters (`BinSnappingEstimator`, `LagReversalEstimator` & `PersistenceEstimator`)
- **Baseline A — `BinSnappingEstimator` (Map Matching Lookup Table)**:
  - **`n_bins`** (`[5, 3, 7]`): Number of discrete integer bins per feature (`5` matches Numerai's `0..4` bins).
  - **`smoothing`** (`[0.0, 25.0, 50.0, 100.0]` / `Real(0.0, 100.0)`): Empirical-Bayes shrinkage weight pulling sparse bin averages toward the global target mean ($\frac{\sum y + m \cdot \bar{y}}{N + m}$).
  - **`agg_mode`** (`['mean', 'corr_weighted']`): `'mean'` averages snapped bin predictions equally across selected features; `'corr_weighted'` weights each feature's bin lookup by $|\text{corr}_j|$.
- **Baseline E — `LagReversalEstimator` (Era Autocorrelation / PACF Sign & Shrinkage Rule)**:
  - **`lag`** (`[1, 2, 4, 6]` / `Integer(1, 6)`): Number of era steps $k$ used to compute each feature's era-level lag-$k$ serial dependence.
  - **`acf_mode`** (`['feature_mean', 'feature_corr']`): `'feature_mean'` computes lag-$k$ dependence on each feature's per-era mean series; `'feature_corr'` computes lag-$k$ dependence on each feature's per-era target-correlation series.
  - **`acf_metric`** (`['acf', 'pacf']`): `'acf'` uses lag-$k$ autocorrelation; `'pacf'` uses lag-$k$ partial autocorrelation (`statsmodels.tsa.stattools.pacf`).
  - **`acf_detrend`** (`[None, 'linear', 'constant']`): Detrends the era series via `scipy.signal.detrend` before computing $\text{ACF}(k)$/$	ext{PACF}(k)$ (`None` = raw series, `'linear'` = removes linear time trend, `'constant'` = subtracts series mean).
  - **`acf_threshold`** (`[-0.05, 0.0, 0.05]` / `Real(-0.20, 0.20)`): Cutoff threshold $\tau$ for sign inversion or continuous shrinkage.
  - **`sign_mode`** (`['hard', 'soft_threshold', 'tanh']`): `'hard'` applies $\pm 1$ sign flip around `acf_threshold`; `'soft_threshold'` applies continuous soft-thresholded shrinkage; `'tanh'` applies smooth $\tanh((\text{acf} - \tau)/0.05)$ shrinkage.
  - **`center_features`** (`[True, False]`): `True` centers each feature by its training mean before applying the signed correlation weights; `False` multiplies raw bin values.
- **Brownlee Persistence Baseline — `PersistenceEstimator` (Decoupled Per-Ticker Target Persistence / Zero-Rule)**:
  - **`ticker_col`** (`[TICKER_COL]`): Security identifier column (`'numerai_ticker'`).
  - **`persistence_mode`** (`['last', 'expanding_mean', None]`): `'last'` predicts each ticker's most recent settled training target; `'expanding_mean'` predicts each ticker's historical training target mean; `None` falls back to Brownlee's Zero-Rule baseline (global training target mean).
  - **`persistence_lag`** (`[4, 1, 2, 8, None]`): Minimum era lag required before a training target is treated as settled (`4` for 20-day targets; `None` uses all available training history).
  - **`smoothing`** (`[0.0, 10.0, 25.0, 50.0]` / `Real(0.0, 100.0)`): Shrinkage pulling ticker target history toward the global training mean (`0.0` or `None` = no shrinkage).

---

- **Evaluation (`utils.evaluate_numerai_predictions`)**: Reports official `numerai_tools.scoring.numerai_corr`, per-era `Spearman Correlation`, `Average Payout` (`(corrs / 0.2).clip(-1, 1)`), `Sharpe Ratio` (`corrs.mean() / corrs.std()`), `Mean Absolute Error (MAE)`, `Forecast Bias` (`mean(y_true - y_pred)`), `Directional Hit Rate` (sign agreement vs. `0.5`), `Era CORR Lag-1 ACF`, `Ljung-Box p-value` (`acorr_ljungbox`), `ARCH-LM p-value` (`het_arch`), `ADF Stationarity p-value` (`adfuller`), `Skewness`, `Excess Kurtosis`, `Jarque-Bera p-value`, `Max Drawdown`, and validation `Feature Exposure`.

In [ ]:
import gc
import os
import sys
import json
import time
import warnings
import importlib
from pathlib import Path
from datetime import datetime

python_env_dir = Path(sys.executable).parent
library_bin_dir = python_env_dir / 'Library' / 'bin'
if library_bin_dir.is_dir() and str(library_bin_dir) not in os.environ.get('PATH', ''):
    os.environ['PATH'] = str(library_bin_dir) + os.pathsep + str(python_env_dir) + os.pathsep + os.environ.get('PATH', '')

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
try:
    import seaborn as sns
except ImportError:
    sns = None
from IPython.display import display

warnings.filterwarnings('ignore')

# Local, Kaggle (/kaggle/input, /kaggle/usr/lib, /kaggle/working), & nested dataset path routing
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir).lower() == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)
if os.path.isdir(src_path) and src_path not in sys.path:
    sys.path.insert(0, src_path)

module_search_roots = [
    Path(workspace_root),
    Path('/kaggle/working'),
    Path('/kaggle/usr/lib'),
    Path('/kaggle/input'),
]
for search_root in module_search_roots:
    if not search_root.is_dir():
        continue
    for target_file in ('afml_utils.py', 'data_utils.py', 'utils.py', 'baseline_utils.py'):
        for found_path in search_root.rglob(target_file):
            parent_dir = str(found_path.parent)
            if parent_dir not in sys.path:
                sys.path.insert(0, parent_dir)

from pandas.plotting import autocorrelation_plot
from scipy.stats import spearmanr
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from statsmodels.graphics.gofplots import qqplot

from numerai_tools.scoring import numerai_corr

try:
    import skopt
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

try:
    import wandb
except ImportError:
    wandb = None

try:
    import afml_utils
    import data_utils
    import utils
    import baseline_utils
except ModuleNotFoundError as import_err:
    mounted_kaggle_inputs = [str(p) for p in sorted(Path('/kaggle/input').iterdir())] if Path('/kaggle/input').is_dir() else []
    raise ModuleNotFoundError(
        f"{import_err}. Could not locate data_utils.py / utils.py / baseline_utils.py in "
        f"workspace_root={workspace_root}, /kaggle/working, /kaggle/usr/lib, or /kaggle/input "
        f"(mounted inputs: {mounted_kaggle_inputs}). "
        "On Kaggle, attach a Dataset or Utility Script containing src/data_utils.py, src/utils.py, and src/baseline_utils.py."
    ) from import_err
importlib.reload(afml_utils)
importlib.reload(data_utils)
importlib.reload(utils)
importlib.reload(baseline_utils)
from data_utils import (
    compute_per_ticker_timeseries_features,
    discover_and_fetch_v6_meta_or_benchmark,
    load_unity_numerai_data,
    load_unity_numerai_data_cached,
)
from utils import (
    NumeraiExtractions,
    build_experiment_setup,
    build_conditional_bayesian_spaces,
    build_quick_results_df,
    candidate_score,
    compute_r1399_churn_retention,
    compute_versioned_candidate_score,
    make_candidate_scorer,
    compute_eda_era_stationarity,
    compute_era_error_envelope,
    configure_error_binning,
    configure_memory_guard,
    prepare_tuned_pipeline,
    compute_feature_exposure,
    compute_feature_target_correlations,
    compute_per_era_mmc,
    compute_per_era_numerai_corr,
    compute_per_era_spearman_corr,
    deduplicate_param_grid,
    drop_constant_dims,
    evaluate_numerai_predictions,
    export_diagnostics_predictions,
    plot_era_error_envelope,
    plot_residuals_vs_actual,
    plot_stake_pnl_dashboard,
    plot_stake_pnl_monte_carlo,
    prepare_bayesian_space,
    quantize_to_5bins,
    reset_cv_metric_log,
    run_numereval_diagnostics,
    sharpe_ratio,
    build_live_staking_scenarios_df,
    compute_deflated_sharpe_ratio,
    compute_probabilistic_sharpe_ratio,
    simulate_live_staking_scenarios,
    simulate_stake_pnl,
    simulate_stake_pnl_monte_carlo,
    walk_forward_predict,
    CombinatorialPurgedEraCV,
    PurgedEraKFold,
    build_era_cv_splits,
    compute_clustered_mda,
    compute_drawdown_and_tuw,
    compute_hhi_concentration,
    compute_lo_adjusted_sharpe,
    compute_min_track_record_length,
    compute_pbo_cscv,
    compute_per_era_fnc,
    compute_strategy_risk,
    neutralize_predictions,
    purge_train_before_validation,
    summarize_afml_backtest_diagnostics,
    build_compact_afml_reference_table,
    build_diagnostics_summary_table,
    build_executive_scorecard_tables,
    plot_candidate_timing_analysis,
    display_compact_dataframe,
    export_winner_scorecard_json,
    plot_afml_backtest_story,
    plot_cumulative_validation_corr,
    plot_rolling_validation_corr,
    plot_top2_residuals_vs_actual,
    build_afml_diagnostics_df,
    strip_validation_predictions,
)
from baseline_utils import (
    BaseNumeraiEstimator,
    BinSnappingEstimator,
    LagReversalEstimator,
    PersistenceEstimator,
    clear_extraction_cache,
    configure_extraction_cache,
    display_baseline_interpretability,
    display_top_cv_candidates,
    extract_baseline_interpretability,
    extract_cv_analysis_artifacts,
    run_baseline,
)

print('Imports loaded successfully')
print(f'workspace_root={workspace_root} | SKOPT_AVAILABLE={SKOPT_AVAILABLE}')


# Setup

In [ ]:
# --- Experiment Knobs (only things you actually change) ---
TARGET_COL = 'target'          # Training target (auto-sets horizon & embargo)
feature_set = 'all'          # 'small', 'medium', or 'all'
era_stride = 'auto'           # 'auto' (matches target horizon: 12 for 60d, 4 for 20d) or explicit int (1, 2, 4, 12)

baseline_mode = ['bin_snapping', 'lag_reversal', 'persistence']  # 'bin_snapping', 'lag_reversal', 'persistence', 'all', comma-separated string, or list
search_mode = 'bayes'          # 'grid' or 'bayes'
score_version = 'v1_skepta'    # 'v0_default', 'v1_skepta', or 'v2_skepta_mmc'
eval_mode = 'both'             # Default=static only; Tuned walk-forward at end
walk_forward_step_eras = 52    # Annual refit (~8 refits over 399 val eras; applied to [Tuned] winner)
cv_strategy = 'purged_kfold'   # 'timeseries', 'purged_kfold' (AFML Ch. 7), or 'cpcv' (AFML Ch. 12)
n_splits = 5
cv_split = 0.8                 # Chronological CV train/test split ratio (used when n_splits == 1)
downsample_mode = 'recent'         # None, 'within_era', 'recent', 'earliest', 'uniform', or 'random_eras'
downsample_frac = 1.0          # Fraction of strided training data to keep in (0.0, 1.0]
val_downsample_mode = 'recent'     # Validation: None (100%), 'same', 'within_era', 'recent', 'earliest', 'uniform', or 'random_eras'
val_downsample_frac = 1.0             # Fraction of validation data to keep in (0.0, 1.0] (None mirrors downsample_frac)
n_iter_persistence = 337  # BayesSearchCV iterations for Persistence
n_iter_bin_snapping = 37    # BayesSearchCV iterations for Bin-Snapping
n_iter_lag_reversal = 137    # BayesSearchCV iterations for Lag-Reversal
n_iter = max(n_iter_bin_snapping, n_iter_lag_reversal, n_iter_persistence)
n_jobs = -1                   # Use all CPU cores for threaded CV, 6-model eval, & walk-forward
save_results = True
save_plots = False             # Save PNG plots to workspace (CSV/JSON still controlled by save_results)
memory_guard_enabled = True    # Explicit RAM fail-safe: set False to switch off
memory_guard_min_free_gb = 3.0 # Keep >=3 GB free RAM or fail candidate without crashing (warnings raised; error_score=0.0)
time_guard_enabled = True      # Explicit per-candidate wall-clock fail-safe: set False to switch off
max_candidate_seconds = 900.0 # Abort slow candidate after 30 min (1800s) without crashing (warnings raised; error_score=0.0)
error_score = 0.0              # Score assigned if a CV fold fit raises outside fail-safe (0.0 warns & continues)
error_bin_mode = 'pct'         # 'pct' (% error + zero-target tol) or 'abs' (raw residual bins)

setup_cfg = build_experiment_setup(    target_col=TARGET_COL,
    feature_set=feature_set,
    era_stride=era_stride,
    model_tag='baselines',
    baseline_mode=baseline_mode,
    search_mode=search_mode,
    score_version=score_version,
    eval_mode=eval_mode,
    walk_forward_step_eras=walk_forward_step_eras,
    cv_strategy=cv_strategy,
    n_splits=n_splits,
    cv_split=cv_split,
    downsample_mode=downsample_mode,
    downsample_frac=downsample_frac,
    val_downsample_mode=val_downsample_mode,
    val_downsample_frac=val_downsample_frac,
    n_iter=n_iter,
    n_jobs=n_jobs,
    save_results=save_results,
    save_plots=save_plots,
    memory_guard_enabled=memory_guard_enabled,
    memory_guard_min_free_gb=memory_guard_min_free_gb,
    time_guard_enabled=time_guard_enabled,
    max_candidate_seconds=max_candidate_seconds,
    error_score=error_score,
    error_bin_mode=error_bin_mode,
    workspace_root=workspace_root,
)
globals().update(setup_cfg)


# Prep

In [ ]:
_prep_t0 = time.perf_counter()
train_df, _, feature_columns, feature_metadata = load_unity_numerai_data_cached(
    data_version=data_version,
    data_folder=data_folder,
    kaggle_dataset=kaggle_dataset,
    feature_set=feature_set,
    target_col=TARGET_COL,
    era_stride=era_stride,
    downsample_mode=downsample_mode,
    downsample_frac=downsample_frac,
    val_downsample_mode=val_downsample_mode,
    val_downsample_frac=val_downsample_frac,
    random_state=random_state,
    ticker_timeseries_features=ticker_timeseries_features,
    ticker_rolling_windows=ticker_rolling_windows,
    ticker_lag_steps=ticker_lag_steps,
    workspace_root=workspace_root,
    auto_download=True,
    use_numerapi_fallback=True,
    force_reload=force_reload_data_cache,
    split='train',
)
validation_df = None
prep_elapsed_s = time.perf_counter() - _prep_t0
print(f'Loaded {len(feature_columns)} int8 features | train={train_df.shape} ({train_df[ERA_COL].nunique()} eras) | validation deferred until post-training evaluation')


## Exploratory Data Analysis (EDA)

In [ ]:
available_targets = list(feature_metadata.get('targets', [TARGET_COL]))
print(f'Available TARGET_COL options ({len(available_targets)}):', available_targets)
print('Available feature_set options:', {k: len(v) for k, v in feature_metadata.get('feature_sets', {}).items()})

print('\nTraining data:')
display(train_df.head(2))
print('Training set info:')
train_df.info()
if validation_df is not None:
    print('\nValidation data:')
    display(validation_df.head(2))
    print('\nValidation set info:')
    validation_df.info()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

train_era_sizes = train_df.groupby(ERA_COL)[TARGET_COL].size()
axes[0].plot(train_era_sizes.index, train_era_sizes.to_numpy(), color='#2563eb', linewidth=1.8, label='Train')
if validation_df is not None:
    val_era_sizes = validation_df.groupby(ERA_COL)[TARGET_COL].size()
    axes[0].plot(val_era_sizes.index, val_era_sizes.to_numpy(), color='#10b981', linewidth=1.8, label='Validation')
axes[0].set_title('Era Sizes Over Time', weight='bold', fontsize=13)
axes[0].set_xlabel('Era')
axes[0].set_ylabel('Rows per Era')
axes[0].legend()

train_feat_stds = train_df[feature_columns].std()
axes[1].hist(train_feat_stds, bins=30, alpha=0.55, density=True, color='#2563eb', label='Train')
if validation_df is not None:
    val_feat_stds = validation_df[feature_columns].std()
    axes[1].hist(val_feat_stds, bins=30, alpha=0.55, density=True, color='#10b981', label='Validation')
axes[1].set_title('Feature Standard Deviation Distribution', weight='bold', fontsize=13)
axes[1].set_xlabel('Standard Deviation')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
input_columns = [ERA_COL] + ([TICKER_COL] if TICKER_COL in train_df.columns else []) + feature_columns
X_train = train_df[input_columns].reset_index(drop=True)
y_train = train_df[TARGET_COL].astype(np.float32).reset_index(drop=True)
train_eras = train_df[ERA_COL].astype(np.int32).reset_index(drop=True)

del train_df
gc.collect()

# Era-level purged CV splitter supporting 'timeseries', 'purged_kfold' (AFML Ch. 7), and 'cpcv' (AFML Ch. 12)
cv_object = build_era_cv_splits(
    train_eras,
    cv_strategy=cv_strategy,
    n_splits=n_splits,
    cv_split=cv_split,
    target_horizon_eras=target_horizon_eras,
    embargo_eras=embargo_eras,
)
print(
    f'Built {len(cv_object)} strictly era-purged CV fold(s) using cv_strategy={cv_strategy!r} '
    f'(horizon={target_horizon_eras} eras, embargo={embargo_eras} eras)'
)


In [ ]:
raw_feature_sets = feature_metadata.get('feature_sets', {})
meta_set_names = {'small', 'medium', 'all'}
active_feature_set_cols = set(feature_columns)
unity_feature_groups_dict = {
    group_name: tuple(col for col in group_cols if col in active_feature_set_cols)
    for group_name, group_cols in raw_feature_sets.items()
    if group_name not in meta_set_names and isinstance(group_cols, list)
}
unity_feature_groups_dict = {k: v for k, v in unity_feature_groups_dict.items() if len(v) >= 2}
unity_feature_groups = tuple(sorted(unity_feature_groups_dict.items())) if unity_feature_groups_dict else None

numerai_extractions = NumeraiExtractions(
    era_col=ERA_COL,
    target_col=TARGET_COL,
    ticker_col=TICKER_COL,
    year_horizon_filter=None,
    variance_threshold=None,
    correlation_method='pandas_mean',
    abs_corr=True,
    top_k=None,
    feature_selection_percentile=None,
    sparse_selector=None,
    sparse_alpha=0.01,
    pca_n_components=None,
    quantize_scope='era',
    quantize_strategy='zscore',
    sequence_top_n=4,
    feature_groups=unity_feature_groups,
    basic_group_stats=None,
    fourier_period_eras=52,
    fourier_order=None,
    rolling_era_windows=(4, 12),
    include_rolling_era=False,
    era_lag_scales=(1, 2, 4),
    include_lag_diff=False,
    ticker_features=None,
    ticker_windows=(4, 12),
    ticker_lag_steps=None,
    era_dispersion_features=None,
    missing_fill_value=2.0,
)

bin_snapping_estimator = BinSnappingEstimator(
    extractor=clone(numerai_extractions),
    top_k=20,
    n_bins=5,
    correlation_method='pandas_mean',
    abs_corr=True,
    smoothing=0.0,
    agg_mode='mean',
    era_weighting=era_weighting,
    recency_decay=recency_decay,
    prediction_transform=None,
    bias_correction=None,
    walk_forward_step_eras=None,
    walk_forward_window=walk_forward_window,
    target_horizon_eras=target_horizon_eras,
    era_stride=1,
    era_col=ERA_COL,
    target_col=TARGET_COL,
    verbose=verbose,
    random_state=random_state,
)

lag_reversal_estimator = LagReversalEstimator(
    extractor=clone(numerai_extractions),
    lag=1,
    acf_mode='feature_mean',
    acf_metric='acf',
    acf_detrend='linear',
    acf_threshold=0.0,
    sign_mode='hard',
    top_k=None,
    correlation_method='pandas_mean',
    abs_corr=True,
    center_features=True,
    era_weighting=era_weighting,
    recency_decay=recency_decay,
    prediction_transform=None,
    bias_correction=None,
    walk_forward_step_eras=None,
    walk_forward_window=walk_forward_window,
    target_horizon_eras=target_horizon_eras,
    era_stride=1,
    era_col=ERA_COL,
    target_col=TARGET_COL,
    verbose=verbose,
    random_state=random_state,
)

persistence_estimator = PersistenceEstimator(
    extractor=None,
    ticker_col=TICKER_COL,
    persistence_mode='last',
    persistence_lag=target_horizon_eras,
    smoothing=0.0,
    top_k=None,
    correlation_method=None,
    abs_corr=True,
    era_weighting=era_weighting,
    recency_decay=recency_decay,
    prediction_transform=None,
    bias_correction=None,
    walk_forward_step_eras=None,
    walk_forward_window=walk_forward_window,
    target_horizon_eras=target_horizon_eras,
    era_stride=1,
    era_col=ERA_COL,
    target_col=TARGET_COL,
    verbose=verbose,
    random_state=random_state,
)

bin_snapping_pipeline = Pipeline([('estimator', bin_snapping_estimator)])
lag_reversal_pipeline = Pipeline([('estimator', lag_reversal_estimator)])
persistence_pipeline = Pipeline([('estimator', persistence_estimator)])
print('Bin-Snapping pipeline ready:', bin_snapping_pipeline)
print('Lag-Reversal pipeline ready:', lag_reversal_pipeline)
print('Persistence pipeline ready:', persistence_pipeline)
del feature_metadata, raw_feature_sets
gc.collect()

# Params

## Grid

In [ ]:
# SHARED `NumeraiExtractions` GRID SPACE (Used by both `BinSnappingEstimator` and `LagReversalEstimator`)
GRID_EXTRACTOR_PARAM_SPACE = {
    # ----------------------------------------------------------
    # EXTRACTOR — Era/Target Routing, Stride, Correlation & Feature Selection
    # ----------------------------------------------------------
    'estimator__extractor__era_col': [ERA_COL],
    'estimator__extractor__ticker_col': [TICKER_COL],
    'estimator__extractor__target_col': [
        TARGET_COL,
        # 'target_cyrusd_20',
        # 'target_teager2b_20',
    ],
    'estimator__extractor__year_horizon_filter': [
        None,
        # (1, 2, 3),
        # (4, 5, 6),
        # (7, 8, 9),
    ],
    'estimator__extractor__variance_threshold': [
        None,
        # 0.02,
        # 0.05,
        # 0.10,
    ],
    'estimator__extractor__correlation_method': [
        'pandas_mean',
        # None,
        # 'numerai_mean',
        # 'pearson',
        # 'spearman'
    ],
    'estimator__extractor__abs_corr': [
        True,
        # False,
    ],
    'estimator__extractor__top_k': [
        None,
        # 10,
        # 20,
        # 40,
    ],
    'estimator__extractor__feature_selection_percentile': [
        None,
        # 25.0,
        # 50.0,
        # 75.0,
        # 100.0,
    ],
    'estimator__extractor__sparse_selector': [
        None,
        # 'lasso',
        # 'elasticnet',
    ],
    'estimator__extractor__sparse_alpha': [
        0.01,
        # 0.001,
        # 0.1,
    ],
    'estimator__extractor__pca_n_components': [
        None,
        # 10,
        # 20,
        # 0.90,
    ],
    'estimator__extractor__quantize_scope': [
        'era',
        # 'train_fit',
        # 'global',
    ],
    'estimator__extractor__quantize_strategy': [
        'zscore',
        # 'quantile',
    ],
    'estimator__extractor__sequence_top_n': [
        4,
        # 2,
        # 8,
    ],
    'estimator__extractor__missing_fill_value': [
        2.0,
        # 0.0,
        # 4.0,
        # 'mean',
        # 'median',
        # 'era_mean',
        # 'era_median',
    ],

    # ----------------------------------------------------------
    # EXTRACTOR — Basic Group Stats, Fourier Harmonics & Era/Ticker Sequence Features
    # ----------------------------------------------------------
    'estimator__extractor__basic_group_stats': [
        None,
        # 'mean',
        # 'std',
        # 'skew',
        # 'mean|std',
        # 'mean|skew',
        # 'std|skew',
        # 'mean|std|skew',
    ],
    'estimator__extractor__fourier_period_eras': [
        52,
        # 26,
        # 13,
    ],
    'estimator__extractor__fourier_order': [
        None,
        # 1,
        # 2,
        # 3,
    ],
    'estimator__extractor__rolling_era_windows': [
        (4, 12),
        # (2, 6, 12),
        # (4, 8, 16),
    ],
    'estimator__extractor__include_rolling_era': [
        False,
        # True,
    ],
    'estimator__extractor__era_lag_scales': [
        (1, 2, 4),
        # (1, 2, 4, 8),
        # (2, 4, 8),
    ],
    'estimator__extractor__include_lag_diff': [
        False,
        # True,
    ],
    'estimator__extractor__ticker_features': [
        None,
        # 'rolling_mean',
        # 'momentum_diff',
        # 'weeks_since_high',
        # 'rolling_rank',
        # 'weeks_since_change',
        # 'expanding_stats',
        # 'raw_lags',
        # 'rolling_mean|momentum_diff',
        # 'rolling_mean|momentum_diff|rolling_rank',
        # 'expanding_stats|raw_lags',
        # 'rolling_mean|momentum_diff|weeks_since_high|rolling_rank|weeks_since_change|expanding_stats|raw_lags',
    ],
    'estimator__extractor__ticker_windows': [
        (4, 12),
        # (2, 6, 12),
    ],
    'estimator__extractor__ticker_lag_steps': [
        None,
        # (1, 2),
        # (1, 2, 4),
    ],
    'estimator__extractor__era_dispersion_features': [
        None,
        # 'mean',
        # 'std',
        # 'extreme_frac',
        # 'mean|std',
        # 'mean|extreme_frac',
        # 'std|extreme_frac',
        # 'mean|std|extreme_frac',
    ],
}

GRID_PARAM_SPACE_BIN_SNAPPING = {
    **GRID_EXTRACTOR_PARAM_SPACE,

    # ----------------------------------------------------------
    # MODEL — Baseline A (`BinSnappingEstimator` — Map Matching Lookup Table)
    # ----------------------------------------------------------
    'estimator__era_col': [ERA_COL],
    'estimator__target_col': [TARGET_COL],
    'estimator__correlation_method': [
        'pandas_mean',
        # None,
        # 'numerai_mean',
        # 'pearson',
        # 'spearman',
    ],
    'estimator__top_k': [
        20,
        # None,
        # 10,
        # 40,
    ],
    'estimator__n_bins': [
        5,
        # 3,
        # 7,
    ],
    'estimator__abs_corr': [
        True,
        # False,
    ],
    'estimator__smoothing': [
        0.0,
        # 25.0,
        # 50.0,
        # 100.0,
    ],
    'estimator__agg_mode': [
        'mean',
        # 'corr_weighted',
    ],
    'estimator__prediction_transform': [
        None,
        # 'era_zscore',
        # 'era_rank',
    ],
    'estimator__bias_correction': [
        None,
        # 'mean_residual',
    ],
    'estimator__walk_forward_step_eras': [
        None,
        # 10,
        # 20,
        # 40,
    ],
    'estimator__walk_forward_window': [
        walk_forward_window,
        # 'expanding',
        # 'sliding',
    ],
    'estimator__target_horizon_eras': [target_horizon_eras],
    'estimator__era_stride': [
        1,
        # 2,
        # 4,
        # 12,
    ],
    'estimator__era_weighting': [
        era_weighting,
        # None,
        # 'equal_era',
        # 'recency_linear',
        # 'recency_exp',
        # 'uniqueness',
    ],
    'estimator__recency_decay': [
        recency_decay,
        # 0.001,
        # 0.010,
    ],
    'estimator__verbose': [verbose],
    'estimator__random_state': [random_state],
}

GRID_PARAM_SPACE_LAG_REVERSAL = {
    **GRID_EXTRACTOR_PARAM_SPACE,

    # ----------------------------------------------------------
    # MODEL — Baseline E (`LagReversalEstimator` — Era Autocorrelation / PACF Sign & Shrinkage Rule)
    # ----------------------------------------------------------
    'estimator__era_col': [ERA_COL],
    'estimator__target_col': [TARGET_COL],
    'estimator__correlation_method': [
        'pandas_mean',
        # None,
        # 'numerai_mean',
        # 'pearson',
        # 'spearman',
    ],
    'estimator__lag': [
        1,
        # 2,
        # 4,
        # 6,
    ],
    'estimator__acf_mode': [
        'feature_mean',
        # 'feature_corr',
    ],
    'estimator__acf_metric': [
        'acf',
        # 'pacf',
    ],
    'estimator__acf_detrend': [
        'linear',
        # None,
        # 'constant',
    ],
    'estimator__acf_threshold': [
        0.0,
        # -0.05,
        # 0.05,
    ],
    'estimator__sign_mode': [
        'hard',
        # 'soft_threshold',
        # 'tanh',
    ],
    'estimator__top_k': [
        None,
        # 10,
        # 20,
        # 40,
    ],
    'estimator__abs_corr': [
        True,
        # False,
    ],
    'estimator__center_features': [
        True,
        # False,
    ],
    'estimator__prediction_transform': [
        None,
        # 'era_zscore',
        # 'era_rank',
    ],
    'estimator__bias_correction': [
        None,
        # 'mean_residual',
    ],
    'estimator__walk_forward_step_eras': [
        None,
        # 10,
        # 20,
        # 40,
    ],
    'estimator__walk_forward_window': [
        walk_forward_window,
        # 'expanding',
        # 'sliding',
    ],
    'estimator__target_horizon_eras': [target_horizon_eras],
    'estimator__era_stride': [
        1,
        # 2,
        # 4,
        # 12,
    ],
    'estimator__era_weighting': [
        era_weighting,
        # None,
        # 'equal_era',
        # 'recency_linear',
        # 'recency_exp',
        # 'uniqueness',
    ],
    'estimator__recency_decay': [
        recency_decay,
        # 0.001,
        # 0.010,
    ],
    'estimator__verbose': [verbose],
    'estimator__random_state': [random_state],
}

# Decoupled Persistence Grid Space (Does NOT inherit `GRID_EXTRACTOR_PARAM_SPACE`)
GRID_PARAM_SPACE_PERSISTENCE = {
    # ----------------------------------------------------------
    # MODEL — Baseline P (`PersistenceEstimator` — Brownlee Target Persistence / Zero-Rule)
    # ----------------------------------------------------------
    'estimator__era_col': [ERA_COL],
    'estimator__target_col': [TARGET_COL],
    'estimator__ticker_col': [TICKER_COL],
    'estimator__persistence_mode': [
        'last',
        # 'expanding_mean',
        # None,
    ],
    'estimator__persistence_lag': [target_horizon_eras],
    'estimator__smoothing': [
        0.0,
        # 10.0,
        # 25.0,
        # 50.0,
    ],
    'estimator__prediction_transform': [
        None,
        # 'era_zscore',
        # 'era_rank',
    ],
    'estimator__bias_correction': [
        None,
        # 'mean_residual',
    ],
    'estimator__walk_forward_step_eras': [
        None,
        # 10,
        # 20,
        # 40,
    ],
    'estimator__walk_forward_window': [
        walk_forward_window,
        # 'expanding',
        # 'sliding',
    ],
    'estimator__target_horizon_eras': [target_horizon_eras],
    'estimator__era_stride': [
        1,
        # 2,
        # 4,
        # 12,
    ],
    'estimator__era_weighting': [
        era_weighting,
        # None,
        # 'equal_era',
        # 'recency_linear',
        # 'recency_exp',
        # 'uniqueness',
    ],
    'estimator__recency_decay': [
        recency_decay,
        # 0.001,
        # 0.010,
    ],
    'estimator__verbose': [verbose],
    'estimator__random_state': [random_state],
}

## Bayesian

In [ ]:
# BAYESIAN SEARCH ONLY
# Shared `BAYESIAN_EXTRACTOR_PARAM_SPACE` is used by both `BinSnappingEstimator` and `LagReversalEstimator`.
# Mode/selection parameters start with `None` (off); boolean `include_*` flags use `[False, True]`.
BAYESIAN_EXTRACTOR_PARAM_SPACE = dict()
BAYESIAN_PARAM_SPACE_BIN_SNAPPING = dict()
BAYESIAN_PARAM_SPACE_LAG_REVERSAL = dict()
BAYESIAN_PARAM_SPACE_PERSISTENCE = dict()
if SKOPT_AVAILABLE:
    BAYESIAN_EXTRACTOR_PARAM_SPACE = {
        # ----------------------------------------------------------
        # EXTRACTOR — Era/Target Routing, Stride, Correlation & Feature Selection
        # ----------------------------------------------------------
        'estimator__extractor__era_col': Categorical([ERA_COL]),
        'estimator__extractor__ticker_col': Categorical([TICKER_COL]),
        'estimator__extractor__target_col': Categorical([
            TARGET_COL,
        ]),
        'estimator__extractor__year_horizon_filter': Categorical([
            None,
            (1, 2, 3),
            (4, 5, 6),
            (7, 8, 9),
        ]),
        'estimator__extractor__variance_threshold': Categorical([
            None,
            0.02,
            0.05,
            0.10,
        ]),
        'estimator__extractor__correlation_method': Categorical([
            None,
            'pandas_mean',
            'numerai_mean',
            'pearson',
            'spearman',
        ]),
        'estimator__extractor__abs_corr': Categorical([True, False]),
        'estimator__extractor__top_k': Categorical([
            None,
            10,
            20,
            40,
        ]),
        'estimator__extractor__feature_selection_percentile': Categorical([
            None,
            25.0,
            50.0,
            75.0,
            100.0,
        ]),
        'estimator__extractor__sparse_selector': Categorical([
            None,
            'lasso',
            'elasticnet',
        ]),
        'estimator__extractor__sparse_alpha': Real(1e-4, 1e-1, prior='log-uniform'),
        'estimator__extractor__pca_n_components': Categorical([
            None,
            10,
            20,
            40,
        ]),
        'estimator__extractor__quantize_scope': Categorical([
            'era',
            'train_fit',
            'global',
        ]),
        'estimator__extractor__quantize_strategy': Categorical([
            'zscore',
            'quantile',
        ]),
        'estimator__extractor__sequence_top_n': Categorical([
            2,
            4,
            8,
        ]),
        'estimator__extractor__missing_fill_value': Categorical([
            2.0,
            0.0,
            4.0,
            'mean',
            'median',
            'era_mean',
            'era_median',
        ]),

        # ----------------------------------------------------------
        # EXTRACTOR — Basic Group Stats, Fourier Harmonics & Era/Ticker Sequence Features
        # ----------------------------------------------------------
        'estimator__extractor__basic_group_stats': Categorical([
            None,
            'mean',
            'std',
            'skew',
            'mean|std',
            'mean|skew',
            'std|skew',
            'mean|std|skew',
        ]),
        'estimator__extractor__fourier_period_eras': Categorical([
            52,
            26,
            13,
        ]),
        'estimator__extractor__fourier_order': Categorical([
            None,
            1,
            2,
            3,
        ]),
        'estimator__extractor__rolling_era_windows': Categorical([
            (4, 12),
            (2, 6, 12),
            (4, 8, 16),
        ]),
        'estimator__extractor__include_rolling_era': Categorical([False, True]),
        'estimator__extractor__era_lag_scales': Categorical([
            (1, 2, 4),
            (1, 2, 4, 8),
            (2, 4, 8),
        ]),
        'estimator__extractor__include_lag_diff': Categorical([False, True]),
        'estimator__extractor__ticker_features': Categorical([
            None,
            'rolling_mean',
            'momentum_diff',
            'weeks_since_high',
            'rolling_rank',
            'weeks_since_change',
            'expanding_stats',
            'raw_lags',
            'rolling_mean|momentum_diff',
            'rolling_mean|momentum_diff|rolling_rank',
            'expanding_stats|raw_lags',
            'rolling_mean|momentum_diff|weeks_since_high|rolling_rank|weeks_since_change|expanding_stats|raw_lags',
        ]),
        'estimator__extractor__ticker_windows': Categorical([
            (4, 12),
            (2, 6, 12),
        ]),
        'estimator__extractor__ticker_lag_steps': Categorical([
            None,
            (1, 2),
            (1, 2, 4),
        ]),
        'estimator__extractor__era_dispersion_features': Categorical([
            None,
            'mean',
            'std',
            'extreme_frac',
            'mean|std',
            'mean|extreme_frac',
            'std|extreme_frac',
            'mean|std|extreme_frac',
        ]),
    }

    BAYESIAN_PARAM_SPACE_BIN_SNAPPING = {
        **BAYESIAN_EXTRACTOR_PARAM_SPACE,

        # ----------------------------------------------------------
        # MODEL — Baseline A (`BinSnappingEstimator` — Map Matching Lookup Table)
        # ----------------------------------------------------------
        'estimator__era_col': Categorical([ERA_COL]),
        'estimator__target_col': Categorical([TARGET_COL]),
        'estimator__correlation_method': Categorical([
            None,
            'pandas_mean',
            'numerai_mean',
            'pearson',
            'spearman',
        ]),
        'estimator__top_k': Categorical([
            None,
            10,
            20,
            40,
        ]),
        'estimator__n_bins': Categorical([
            5,
            3,
            7,
        ]),
        'estimator__abs_corr': Categorical([True, False]),
        'estimator__smoothing': Real(0.0, 100.0),
        'estimator__agg_mode': Categorical(['mean', 'corr_weighted']),
        'estimator__prediction_transform': Categorical([
            None,
            'era_zscore',
            'era_rank',
        ]),
        'estimator__bias_correction': Categorical([
            None,
            'mean_residual',
        ]),
        'estimator__walk_forward_step_eras': Categorical([None]),
        'estimator__walk_forward_window': Categorical([walk_forward_window]),
        'estimator__target_horizon_eras': Categorical([target_horizon_eras]),
        'estimator__era_stride': Categorical([1]),
        'estimator__era_weighting': Categorical([
            None,
            'equal_era',
            'recency_linear',
            'recency_exp',
            'uniqueness',
            'uniqueness_decay',
        ]),
        'estimator__recency_decay': Real(0.001, 0.020),
        'estimator__verbose': Categorical([verbose]),
        'estimator__random_state': Categorical([random_state]),
    }
    BAYESIAN_PARAM_SPACE_BIN_SNAPPING = build_conditional_bayesian_spaces(
        drop_constant_dims(BAYESIAN_PARAM_SPACE_BIN_SNAPPING, bin_snapping_pipeline),
        total_n_iter=n_iter_bin_snapping,
    )

    BAYESIAN_PARAM_SPACE_LAG_REVERSAL = {
        **BAYESIAN_EXTRACTOR_PARAM_SPACE,

        # ----------------------------------------------------------
        # MODEL — Baseline E (`LagReversalEstimator` — Era Autocorrelation / PACF Sign & Shrinkage Rule)
        # ----------------------------------------------------------
        'estimator__era_col': Categorical([ERA_COL]),
        'estimator__target_col': Categorical([TARGET_COL]),
        'estimator__correlation_method': Categorical([
            None,
            'pandas_mean',
            'numerai_mean',
            'pearson',
            'spearman',
        ]),
        'estimator__lag': Integer(1, 6),
        'estimator__acf_mode': Categorical(['feature_mean', 'feature_corr']),
        'estimator__acf_metric': Categorical(['acf', 'pacf']),
        'estimator__acf_detrend': Categorical([
            None,
            'linear',
            'constant',
        ]),
        'estimator__acf_threshold': Real(-0.20, 0.20),
        'estimator__sign_mode': Categorical([
            'hard',
            'soft_threshold',
            'tanh',
        ]),
        'estimator__top_k': Categorical([
            None,
            10,
            20,
            40,
        ]),
        'estimator__abs_corr': Categorical([True, False]),
        'estimator__center_features': Categorical([True, False]),
        'estimator__prediction_transform': Categorical([
            None,
            'era_zscore',
            'era_rank',
        ]),
        'estimator__bias_correction': Categorical([
            None,
            'mean_residual',
        ]),
        'estimator__walk_forward_step_eras': Categorical([None]),
        'estimator__walk_forward_window': Categorical([walk_forward_window]),
        'estimator__target_horizon_eras': Categorical([target_horizon_eras]),
        'estimator__era_stride': Categorical([1]),
        'estimator__era_weighting': Categorical([
            None,
            'equal_era',
            'recency_linear',
            'recency_exp',
            'uniqueness',
            'uniqueness_decay',
        ]),
        'estimator__recency_decay': Real(0.001, 0.020),
        'estimator__verbose': Categorical([verbose]),
        'estimator__random_state': Categorical([random_state]),
    }
    BAYESIAN_PARAM_SPACE_LAG_REVERSAL = build_conditional_bayesian_spaces(
        drop_constant_dims(BAYESIAN_PARAM_SPACE_LAG_REVERSAL, lag_reversal_pipeline),
        total_n_iter=n_iter_lag_reversal,
    )

    # Decoupled Persistence Bayesian Space (Does NOT inherit `BAYESIAN_EXTRACTOR_PARAM_SPACE`)
    BAYESIAN_PARAM_SPACE_PERSISTENCE = {
        # ----------------------------------------------------------
        # MODEL — Baseline P (`PersistenceEstimator` — Brownlee Target Persistence / Zero-Rule)
        # ----------------------------------------------------------
        'estimator__era_col': Categorical([ERA_COL]),
        'estimator__target_col': Categorical([TARGET_COL]),
        'estimator__ticker_col': Categorical([TICKER_COL]),
        'estimator__persistence_mode': Categorical([
            None,
            'last',
            'expanding_mean',
        ]),
        'estimator__persistence_lag': Categorical([target_horizon_eras, target_horizon_eras * 2, target_horizon_eras * 3]),
        'estimator__smoothing': Real(0.0, 100.0),
        'estimator__prediction_transform': Categorical([
            None,
            'era_zscore',
            'era_rank',
        ]),
        'estimator__bias_correction': Categorical([
            None,
            'mean_residual',
        ]),
        'estimator__walk_forward_step_eras': Categorical([None]),
        'estimator__walk_forward_window': Categorical([walk_forward_window]),
        'estimator__target_horizon_eras': Categorical([target_horizon_eras]),
        'estimator__era_stride': Categorical([1]),
        'estimator__era_weighting': Categorical([
            None,
            'equal_era',
            'recency_linear',
            'recency_exp',
            'uniqueness',
            'uniqueness_decay',
        ]),
        'estimator__recency_decay': Real(0.001, 0.020),
        'estimator__verbose': Categorical([verbose]),
        'estimator__random_state': Categorical([random_state]),
    }
    BAYESIAN_PARAM_SPACE_PERSISTENCE = build_conditional_bayesian_spaces(
        drop_constant_dims(BAYESIAN_PARAM_SPACE_PERSISTENCE, persistence_pipeline),
        total_n_iter=n_iter_persistence,
    )

bin_snapping_param_names = set(bin_snapping_pipeline.get_params(deep=True))
lag_reversal_param_names = set(lag_reversal_pipeline.get_params(deep=True))
persistence_param_names = set(persistence_pipeline.get_params(deep=True))
assert set(GRID_PARAM_SPACE_BIN_SNAPPING).issubset(bin_snapping_param_names)
assert set(GRID_PARAM_SPACE_LAG_REVERSAL).issubset(lag_reversal_param_names)
assert set(GRID_PARAM_SPACE_PERSISTENCE).issubset(persistence_param_names)
for _bspace, _pnames in (
    (BAYESIAN_PARAM_SPACE_BIN_SNAPPING, bin_snapping_param_names),
    (BAYESIAN_PARAM_SPACE_LAG_REVERSAL, lag_reversal_param_names),
    (BAYESIAN_PARAM_SPACE_PERSISTENCE, persistence_param_names),
):
    _bkeys = set().union(*(sub.keys() for sub, _ in _bspace)) if isinstance(_bspace, list) else set(_bspace)
    assert _bkeys.issubset(_pnames)

if search_mode == 'bayes':
    bin_snapping_param_space = BAYESIAN_PARAM_SPACE_BIN_SNAPPING
    lag_reversal_param_space = BAYESIAN_PARAM_SPACE_LAG_REVERSAL
    persistence_param_space = BAYESIAN_PARAM_SPACE_PERSISTENCE
else:
    bin_snapping_param_space = GRID_PARAM_SPACE_BIN_SNAPPING
    lag_reversal_param_space = GRID_PARAM_SPACE_LAG_REVERSAL
    persistence_param_space = GRID_PARAM_SPACE_PERSISTENCE
print(f'Active search_mode={search_mode} | baseline_mode={baseline_mode}')

# Train

In [ ]:
_train_t0 = time.perf_counter()
baseline_results = {}
search_summary_rows = []
cv_results_by_baseline = {}
baseline_interpretability = {}

configure_error_binning(
    mode=error_bin_mode,
    pct_bins=error_pct_bins,
    zero_tols=error_zero_tols,
    abs_bins=error_abs_bins,
)
configure_memory_guard(enabled=memory_guard_enabled, min_free_gb=memory_guard_min_free_gb)

effective_cache_max_entries = max(int(extraction_cache_max_entries), int(n_splits) if int(n_splits) > 1 else 1)
configure_extraction_cache(enabled=extraction_cache_enabled, max_entries=effective_cache_max_entries)

active_baselines = list(selected_baselines)
all_baseline_jobs = []
if 'persistence' in active_baselines:
    all_baseline_jobs.append(('Persistence', persistence_pipeline, persistence_param_space, n_iter_persistence, True))
if 'bin_snapping' in active_baselines:
    all_baseline_jobs.append(('Bin-Snapping', bin_snapping_pipeline, bin_snapping_param_space, n_iter_bin_snapping, False))
if 'lag_reversal' in active_baselines:
    all_baseline_jobs.append(('Lag-Reversal', lag_reversal_pipeline, lag_reversal_param_space, n_iter_lag_reversal, False))

# =====================================================================
# HALF 1: Train & Tune all 3 baselines (6 fitted models: 3 Default + 3 Tuned)
# =====================================================================
val_min_era_ref = (
    [int(feature_metadata['val_min_era'])]
    if ('val_min_era' in feature_metadata and feature_metadata['val_min_era'] is not None)
    else [int(train_eras.max()) + int(target_horizon_eras)]
)
X_train_static, y_train_static, train_eras_static, static_purge_info = purge_train_before_validation(
    X_train,
    y_train,
    train_eras,
    val_min_era_ref,
    target_horizon_eras=target_horizon_eras,
)
if static_purge_info['n_eras_purged'] > 0:
    print(
        f"  [AFML Purge | Baselines] Purged {static_purge_info['n_eras_purged']} trailing train era(s) "
        f"(cutoff <= era {static_purge_info['cutoff_era']}, min_val_era={static_purge_info['min_val_era']}, "
        f"horizon={target_horizon_eras}) before static fits."
    )

persist_cols = [c for c in (ERA_COL, TICKER_COL) if c in X_train.columns]
trained_baseline_records = []

for b_name, b_pipe, b_space, b_n_iter, meta_only in all_baseline_jobs:
    X_tr_in = X_train[persist_cols] if meta_only else X_train
    X_tr_st_in = X_train_static[persist_cols] if meta_only else X_train_static
    f_cols_in = [] if meta_only else feature_columns

    # 1) Default model fit + train scoring
    default_model = clone(b_pipe).fit(X_tr_st_in, y_train_static)
    default_params = dict(default_model.named_steps['estimator'].get_params())
    default_train_preds = default_model.predict(X_tr_st_in)
    default_train_eval = evaluate_numerai_predictions(
        y_train_static,
        default_train_preds,
        eras=train_eras_static,
        features_df=X_tr_st_in,
        feature_columns=f_cols_in,
        label=f'{b_name} — Final Training Scores',
        verbose=True,
    )
    del default_train_preds

    # 2) CV search
    clear_extraction_cache()
    cv_scorer = make_candidate_scorer(score_version=score_version)
    if search_mode == 'bayes':
        if not SKOPT_AVAILABLE or BayesSearchCV is None:
            raise ImportError('scikit-optimize is required for Bayesian search')
        b_spaces = (
            b_space
            if isinstance(b_space, list)
            else build_conditional_bayesian_spaces(b_space, total_n_iter=b_n_iter)
        )
        search = BayesSearchCV(
            estimator=b_pipe,
            search_spaces=b_spaces,
            n_iter=b_n_iter,
            optimizer_kwargs={'base_estimator': 'ET', 'acq_func': 'EI', 'n_initial_points': min(4, max(1, b_n_iter // 3))},
            scoring=cv_scorer,
            cv=cv_object,
            refit=False,
            random_state=random_state,
            verbose=verbose,
            error_score=error_score,
            n_jobs=1,
        )
    else:
        deduped_grid = deduplicate_param_grid(b_space, base_params=b_pipe.get_params(deep=True))
        search = GridSearchCV(
            estimator=b_pipe,
            param_grid=deduped_grid,
            scoring=cv_scorer,
            cv=cv_object,
            refit=False,
            verbose=verbose,
            error_score=error_score,
            n_jobs=1,
        )

    _base_cv_jobs = min(os.cpu_count() or 1, max(1, len(cv_object))) if (int(n_jobs) != 1) else 1
    search.set_params(n_jobs=_base_cv_jobs)
    reset_cv_metric_log()
    if _base_cv_jobs > 1:
        from joblib import parallel_backend
        with parallel_backend('threading'):
            search.fit(X_tr_in, y_train, groups=train_eras)
    else:
        search.fit(X_tr_in, y_train, groups=train_eras)
    cv_results_df = build_quick_results_df(search)
    cv_results_by_baseline[b_name] = cv_results_df
    trials_matrix = cv_results_df.attrs.get('trials_era_corrs_df')
    pbo_info = compute_pbo_cscv(trials_matrix, random_state=random_state) if trials_matrix is not None else {'pbo': float(np.nan)}
    effective_ver = (
        str(cv_results_df.loc[0, 'score_version'])
        if ('score_version' in cv_results_df.columns and len(cv_results_df) > 0)
        else str(score_version)
    )
    best_params = dict(search.best_params_)
    search_summary_rows.append({
        'baseline': b_name,
        'search_mode': search_mode,
        'score_version': effective_ver,
        'best_cv_score': float(search.best_score_),
        'best_cv_corr': float(cv_results_df.loc[0, 'mean_numerai_corr'])
        if ('mean_numerai_corr' in cv_results_df.columns and len(cv_results_df) > 0)
        else float(search.best_score_),
        'best_cv_score_pct': round(float(search.best_score_) * 100.0, 2),
        'pbo_cscv': round(float(pbo_info['pbo']), 4) if np.isfinite(pbo_info['pbo']) else float(np.nan),
        'best_params': best_params,
    })

    # 3) Tuned refit + train scoring + interpretability
    tuned_pipeline = prepare_tuned_pipeline(b_pipe, best_params)
    tuned_model = clone(tuned_pipeline).fit(X_tr_st_in, y_train_static)
    tuned_params = dict(tuned_model.named_steps['estimator'].get_params())
    tuned_train_preds = tuned_model.predict(X_tr_st_in)
    tuned_train_eval = evaluate_numerai_predictions(
        y_train_static,
        tuned_train_preds,
        eras=train_eras_static,
        features_df=X_tr_st_in,
        feature_columns=f_cols_in,
        label=f'{b_name} [Tuned] — Final Training Scores',
        verbose=True,
    )
    baseline_interpretability.update(extract_baseline_interpretability({b_name: tuned_model}))
    trial_srs = (
        cv_results_df['mean_sharpe'].dropna().to_numpy(dtype=np.float64)
        if ('mean_sharpe' in cv_results_df.columns and len(cv_results_df) > 0)
        else None
    )
    trained_baseline_records.append({
        'name': b_name,
        'meta_only': meta_only,
        'effective_ver': effective_ver,
        'default_model': default_model,
        'default_params': default_params,
        'default_train_eval': default_train_eval,
        'tuned_model': tuned_model,
        'tuned_pipeline': tuned_pipeline,
        'tuned_params': tuned_params,
        'tuned_train_eval': tuned_train_eval,
        'trial_srs': trial_srs,
        'trials_matrix': trials_matrix,
    })
    del tuned_train_preds, search
    clear_extraction_cache()
    gc.collect()

# Free all training data before loading validation
del X_train, y_train, train_eras, X_train_static, y_train_static, train_eras_static
clear_extraction_cache()
gc.collect()

# =====================================================================
# HALF 2: Load Validation once, evaluate all 6 models (3 Default + 3 Tuned), Walk-Forward
# =====================================================================
_, validation_df, _, _ = load_unity_numerai_data_cached(
    data_version=data_version,
    data_folder=data_folder,
    kaggle_dataset=kaggle_dataset,
    feature_set=feature_set,
    target_col=TARGET_COL,
    era_stride=era_stride,
    downsample_mode=downsample_mode,
    downsample_frac=downsample_frac,
    val_downsample_mode=val_downsample_mode,
    val_downsample_frac=val_downsample_frac,
    random_state=random_state,
    ticker_timeseries_features=ticker_timeseries_features,
    ticker_rolling_windows=ticker_rolling_windows,
    ticker_lag_steps=ticker_lag_steps,
    workspace_root=workspace_root,
    auto_download=True,
    use_numerapi_fallback=True,
    force_reload=False,
    split='validation',
)
val_input_columns = [ERA_COL] + ([TICKER_COL] if TICKER_COL in validation_df.columns else []) + feature_columns
X_validation = validation_df[val_input_columns].reset_index(drop=True)
y_validation = validation_df[TARGET_COL].astype(np.float32).reset_index(drop=True)
validation_eras = validation_df[ERA_COL].astype(np.int32).reset_index(drop=True)
val_diag_meta = validation_df[[c for c in ('id', 'numerai_ticker', 'date', ERA_COL) if c in validation_df.columns]].reset_index(drop=True).copy()
val_era_dates = (
    val_diag_meta.groupby(ERA_COL, sort=True)['date'].first()
    if ('date' in val_diag_meta.columns and ERA_COL in val_diag_meta.columns)
    else None
)
del validation_df
gc.collect()

from concurrent.futures import ThreadPoolExecutor as _BaseEvalPool

def _eval_baseline_pair(rec):
    b_name = rec['name']
    meta_only = rec['meta_only']
    X_val_in = X_validation[persist_cols] if meta_only else X_validation
    f_cols_in = [] if meta_only else feature_columns
    _def_p = rec['default_model'].predict(X_val_in)
    _def_ev = evaluate_numerai_predictions(
        y_validation, _def_p, eras=validation_eras, features_df=X_val_in,
        feature_columns=f_cols_in, label=f'{b_name} — Final Validation Scores', verbose=True,
    )
    _tun_p = rec['tuned_model'].predict(X_val_in)
    _tun_ev = evaluate_numerai_predictions(
        y_validation, _tun_p, eras=validation_eras, features_df=X_val_in,
        feature_columns=f_cols_in, label=f'{b_name} [Tuned] — Final Validation Scores', verbose=True,
    )
    return rec, _def_p, _def_ev, _tun_p, _tun_ev

_n_base_eval_workers = min(os.cpu_count() or 1, max(1, len(trained_baseline_records))) if int(n_jobs) != 1 else 1
if _n_base_eval_workers > 1:
    with _BaseEvalPool(max_workers=_n_base_eval_workers) as _eval_pool:
        _eval_outputs = list(_eval_pool.map(_eval_baseline_pair, trained_baseline_records))
else:
    _eval_outputs = [_eval_baseline_pair(r) for r in trained_baseline_records]

for rec, val_preds, val_eval, tuned_val_preds, tuned_val_eval in _eval_outputs:
    b_name = rec['name']
    if eval_mode in ('static', 'both'):
        val_pnl = simulate_stake_pnl(val_eval['numerai_era_corrs'], era_dates=val_era_dates)
        val_dsr_info = compute_deflated_sharpe_ratio(val_eval['numerai_era_corrs'], [val_eval['numerai_sharpe']])
        v_name = f'{b_name} [Default]'
        baseline_results[v_name] = {
            'variant': v_name,
            'score_version': str(score_version),
            'train_corr': rec['default_train_eval']['numerai_corr'],
            'validation_corr': val_eval['numerai_corr'],
            'validation_corr_pct': val_eval['numerai_corr_pct'],
            'positive_era_pct': val_eval['positive_era_pct'],
            'era_consistency_pct': val_eval['era_consistency_pct'],
            'directional_accuracy_pct': val_eval['directional_accuracy_pct'],
            'validation_spearman': val_eval['spearman'],
            'validation_payout': val_eval['payout'],
            'validation_sharpe': val_eval['numerai_sharpe'],
            'validation_lo_sharpe': val_eval['lo_sharpe'],
            'validation_psr': round(float(val_dsr_info['psr']), 4),
            'validation_dsr': round(float(val_dsr_info['dsr']), 4),
            'effective_n_trials_onc': round(float(val_dsr_info['effective_n_trials']), 2),
            'validation_fnc': val_eval['feature_neutral_corr'],
            'validation_fnc_sharpe': val_eval['fnc_sharpe'],
            'tuw_max_eras': val_eval['tuw_max_eras'],
            'hhi_pos': val_eval['hhi_pos'],
            'hhi_neg': val_eval['hhi_neg'],
            'validation_spearman_sharpe': val_eval['spearman_sharpe'],
            'validation_mae': val_eval['mae'],
            'forecast_bias': val_eval['forecast_bias'],
            'directional_hit_rate': val_eval['directional_hit_rate'],
            'era_corr_lag1_acf': val_eval['era_corr_lag1_acf'],
            'ljung_box_pvalue': val_eval['ljung_box_pvalue'],
            'arch_lm_pvalue': val_eval['arch_lm_pvalue'],
            'adf_pvalue': val_eval['adf_pvalue'],
            'era_corr_skew': val_eval['era_corr_skew'],
            'era_corr_kurtosis': val_eval['era_corr_kurtosis'],
            'jarque_bera_pvalue': val_eval['jarque_bera_pvalue'],
            'feature_exposure': val_eval['feature_exposure'],
            'corr_drawdown': val_eval['corr_drawdown'],
            'start_capital': round(float(val_pnl['start_capital']), 2),
            'end_capital': round(float(val_pnl['end_capital']), 2),
            'total_profit': round(float(val_pnl['total_profit']), 2),
            'total_return_pct': round(float(val_pnl['total_return_pct']), 2),
            'money_drawdown_pct': round(float(val_pnl['money_drawdown_pct']), 2),
            'max_drawdown_gbp': round(float(val_pnl['max_drawdown_gbp']), 2),
            'worst_round_gbp': round(float(val_pnl['worst_round_gbp']), 2),
            'corr_gap': rec['default_train_eval']['numerai_corr'] - val_eval['numerai_corr'],
            'sanity_status': val_eval['sanity_status'],
            'validation_preds': val_preds,
            'validation_era_corrs': val_eval['numerai_era_corrs'],
            'params': rec['default_params'],
        }
    rec['default_model'] = None

    eff_trial_srs = rec['trial_srs'] if rec['trial_srs'] is not None else np.array([tuned_val_eval['numerai_sharpe']], dtype=np.float64)
    rec['eff_trial_srs'] = eff_trial_srs
    if eval_mode in ('static', 'both'):
        tuned_val_pnl = simulate_stake_pnl(tuned_val_eval['numerai_era_corrs'], era_dates=val_era_dates)
        tuned_dsr_info = compute_deflated_sharpe_ratio(
            tuned_val_eval['numerai_era_corrs'],
            eff_trial_srs,
            trials_matrix=rec['trials_matrix'],
            random_state=random_state,
        )
        t_name = f'{b_name} [Tuned]'
        baseline_results[t_name] = {
            'variant': t_name,
            'score_version': rec['effective_ver'],
            'train_corr': rec['tuned_train_eval']['numerai_corr'],
            'validation_corr': tuned_val_eval['numerai_corr'],
            'validation_corr_pct': tuned_val_eval['numerai_corr_pct'],
            'positive_era_pct': tuned_val_eval['positive_era_pct'],
            'era_consistency_pct': tuned_val_eval['era_consistency_pct'],
            'directional_accuracy_pct': tuned_val_eval['directional_accuracy_pct'],
            'validation_spearman': tuned_val_eval['spearman'],
            'validation_payout': tuned_val_eval['payout'],
            'validation_sharpe': tuned_val_eval['numerai_sharpe'],
            'validation_lo_sharpe': tuned_val_eval['lo_sharpe'],
            'validation_psr': round(float(tuned_dsr_info['psr']), 4),
            'validation_dsr': round(float(tuned_dsr_info['dsr']), 4),
            'effective_n_trials_onc': round(float(tuned_dsr_info['effective_n_trials']), 2),
            'validation_fnc': tuned_val_eval['feature_neutral_corr'],
            'validation_fnc_sharpe': tuned_val_eval['fnc_sharpe'],
            'tuw_max_eras': tuned_val_eval['tuw_max_eras'],
            'hhi_pos': tuned_val_eval['hhi_pos'],
            'hhi_neg': tuned_val_eval['hhi_neg'],
            'validation_spearman_sharpe': tuned_val_eval['spearman_sharpe'],
            'validation_mae': tuned_val_eval['mae'],
            'forecast_bias': tuned_val_eval['forecast_bias'],
            'directional_hit_rate': tuned_val_eval['directional_hit_rate'],
            'era_corr_lag1_acf': tuned_wf_eval['era_corr_lag1_acf'],
            'ljung_box_pvalue': tuned_val_eval['ljung_box_pvalue'],
            'arch_lm_pvalue': tuned_val_eval['arch_lm_pvalue'],
            'adf_pvalue': tuned_val_eval['adf_pvalue'],
            'era_corr_skew': tuned_val_eval['era_corr_skew'],
            'era_corr_kurtosis': tuned_val_eval['era_corr_kurtosis'],
            'jarque_bera_pvalue': tuned_val_eval['jarque_bera_pvalue'],
            'feature_exposure': tuned_val_eval['feature_exposure'],
            'corr_drawdown': tuned_val_eval['corr_drawdown'],
            'start_capital': round(float(tuned_val_pnl['start_capital']), 2),
            'end_capital': round(float(tuned_val_pnl['end_capital']), 2),
            'total_profit': round(float(tuned_val_pnl['total_profit']), 2),
            'total_return_pct': round(float(tuned_val_pnl['total_return_pct']), 2),
            'money_drawdown_pct': round(float(tuned_val_pnl['money_drawdown_pct']), 2),
            'max_drawdown_gbp': round(float(tuned_val_pnl['max_drawdown_gbp']), 2),
            'worst_round_gbp': round(float(tuned_val_pnl['worst_round_gbp']), 2),
            'corr_gap': rec['tuned_train_eval']['numerai_corr'] - tuned_val_eval['numerai_corr'],
            'sanity_status': tuned_val_eval['sanity_status'],
            'validation_preds': tuned_val_preds,
            'validation_era_corrs': tuned_val_eval['numerai_era_corrs'],
            'params': rec['tuned_params'],
        }
    rec['tuned_model'] = None
    gc.collect()

if eval_mode in ('walk_forward', 'both'):
    train_df, _, _, _ = load_unity_numerai_data_cached(
        data_version=data_version,
        data_folder=data_folder,
        kaggle_dataset=kaggle_dataset,
        feature_set=feature_set,
        target_col=TARGET_COL,
        era_stride=era_stride,
        downsample_mode=downsample_mode,
        downsample_frac=downsample_frac,
        val_downsample_mode=val_downsample_mode,
        val_downsample_frac=val_downsample_frac,
        random_state=random_state,
        ticker_timeseries_features=ticker_timeseries_features,
        ticker_rolling_windows=ticker_rolling_windows,
        ticker_lag_steps=ticker_lag_steps,
        workspace_root=workspace_root,
        auto_download=True,
        use_numerapi_fallback=True,
        force_reload=False,
        split='train',
    )
    X_train = train_df[val_input_columns].reset_index(drop=True)
    y_train = train_df[TARGET_COL].astype(np.float32).reset_index(drop=True)
    train_eras = train_df[ERA_COL].astype(np.int32).reset_index(drop=True)
    del train_df
    gc.collect()

    for rec in trained_baseline_records:
        b_name = rec['name']
        meta_only = rec['meta_only']
        X_tr_in = X_train[persist_cols] if meta_only else X_train
        X_val_in = X_validation[persist_cols] if meta_only else X_validation
        f_cols_in = [] if meta_only else feature_columns

        tuned_wf_preds = walk_forward_predict(
            rec['tuned_pipeline'],
            X_tr_in,
            y_train,
            train_eras,
            X_val_in,
            y_validation,
            validation_eras,
            step_eras=walk_forward_step_eras,
            window_mode=walk_forward_window,
            target_horizon_eras=target_horizon_eras,
            era_stride=era_stride,
            n_jobs=n_jobs,
        )
        tuned_wf_eval = evaluate_numerai_predictions(
            y_validation,
            tuned_wf_preds,
            eras=validation_eras,
            features_df=X_val_in,
            feature_columns=f_cols_in,
            label=f'{b_name} [Tuned Walk-Forward] — Walk-Forward Validation Scores',
            verbose=True,
        )
        tuned_wf_pnl = simulate_stake_pnl(tuned_wf_eval['numerai_era_corrs'], era_dates=val_era_dates)
        tuned_wf_dsr_info = compute_deflated_sharpe_ratio(
            tuned_wf_eval['numerai_era_corrs'],
            rec['eff_trial_srs'],
            trials_matrix=rec['trials_matrix'],
            random_state=random_state,
        )
        wf_name = f'{b_name} [Tuned Walk-Forward]'
        baseline_results[wf_name] = {
            'variant': wf_name,
            'score_version': rec['effective_ver'],
            'train_corr': rec['tuned_train_eval']['numerai_corr'],
            'validation_corr': tuned_wf_eval['numerai_corr'],
            'validation_corr_pct': tuned_wf_eval['numerai_corr_pct'],
            'positive_era_pct': tuned_wf_eval['positive_era_pct'],
            'era_consistency_pct': tuned_wf_eval['era_consistency_pct'],
            'directional_accuracy_pct': tuned_wf_eval['directional_accuracy_pct'],
            'validation_spearman': tuned_wf_eval['spearman'],
            'validation_payout': tuned_wf_eval['payout'],
            'validation_sharpe': tuned_wf_eval['numerai_sharpe'],
            'validation_lo_sharpe': tuned_wf_eval['lo_sharpe'],
            'validation_psr': round(float(tuned_wf_dsr_info['psr']), 4),
            'validation_dsr': round(float(tuned_wf_dsr_info['dsr']), 4),
            'effective_n_trials_onc': round(float(tuned_wf_dsr_info['effective_n_trials']), 2),
            'validation_fnc': tuned_wf_eval['feature_neutral_corr'],
            'validation_fnc_sharpe': tuned_wf_eval['fnc_sharpe'],
            'tuw_max_eras': tuned_wf_eval['tuw_max_eras'],
            'hhi_pos': tuned_wf_eval['hhi_pos'],
            'hhi_neg': tuned_wf_eval['hhi_neg'],
            'validation_spearman_sharpe': tuned_wf_eval['spearman_sharpe'],
            'validation_mae': tuned_wf_eval['mae'],
            'forecast_bias': tuned_wf_eval['forecast_bias'],
            'directional_hit_rate': tuned_wf_eval['directional_hit_rate'],
            'era_corr_lag1_acf': tuned_wf_eval['era_corr_lag1_acf'],
            'ljung_box_pvalue': tuned_wf_eval['ljung_box_pvalue'],
            'arch_lm_pvalue': tuned_wf_eval['arch_lm_pvalue'],
            'adf_pvalue': tuned_wf_eval['adf_pvalue'],
            'era_corr_skew': tuned_wf_eval['era_corr_skew'],
            'era_corr_kurtosis': tuned_wf_eval['era_corr_kurtosis'],
            'jarque_bera_pvalue': tuned_wf_eval['jarque_bera_pvalue'],
            'feature_exposure': tuned_wf_eval['feature_exposure'],
            'corr_drawdown': tuned_wf_eval['corr_drawdown'],
            'start_capital': round(float(tuned_wf_pnl['start_capital']), 2),
            'end_capital': round(float(tuned_wf_pnl['end_capital']), 2),
            'total_profit': round(float(tuned_wf_pnl['total_profit']), 2),
            'total_return_pct': round(float(tuned_wf_pnl['total_return_pct']), 2),
            'money_drawdown_pct': round(float(tuned_wf_pnl['money_drawdown_pct']), 2),
            'max_drawdown_gbp': round(float(tuned_wf_pnl['max_drawdown_gbp']), 2),
            'worst_round_gbp': round(float(tuned_wf_pnl['worst_round_gbp']), 2),
            'corr_gap': rec['tuned_train_eval']['numerai_corr'] - tuned_wf_eval['numerai_corr'],
            'sanity_status': tuned_wf_eval['sanity_status'],
            'validation_preds': tuned_wf_preds,
            'validation_era_corrs': tuned_wf_eval['numerai_era_corrs'],
            'params': rec['tuned_params'],
        }
    del X_train, y_train, train_eras
    gc.collect()

train_elapsed_s = time.perf_counter() - _train_t0
print(f'Completed {len(trained_baseline_records)} baseline(s) ({len(baseline_results)} variants) in {train_elapsed_s:.1f}s')
del X_validation, trained_baseline_records
clear_extraction_cache()
gc.collect()
for _stale_var in ('cv_results_df', 'trials_matrix', 'val_preds', 'tuned_val_preds', 'tuned_wf_preds', 'val_eval', 'tuned_val_eval', 'tuned_wf_eval', 'val_pnl', 'tuned_val_pnl', 'tuned_wf_pnl', 'val_dsr_info'):
    globals().pop(_stale_var, None)
gc.collect()


In [ ]:
comparison_columns = [
    'variant',
    'score_version',
    'train_corr',
    'validation_corr',
    'validation_spearman',
    'validation_payout',
    'era_consistency_pct',
    'validation_sharpe',
    'validation_lo_sharpe',
    'validation_psr',
    'validation_dsr',
    'effective_n_trials_onc',
    'validation_fnc',
    'validation_fnc_sharpe',
    'tuw_max_eras',
    'hhi_pos',
    'hhi_neg',
    'validation_spearman_sharpe',
    'validation_mae',
    'forecast_bias',
    'directional_hit_rate',
    'era_corr_lag1_acf',
    'ljung_box_pvalue',
    'arch_lm_pvalue',
    'adf_pvalue',
    'era_corr_skew',
    'era_corr_kurtosis',
    'jarque_bera_pvalue',
    'feature_exposure',
    'corr_drawdown',
    'start_capital',
    'end_capital',
    'total_profit',
    'total_return_pct',
    'money_drawdown_pct',
    'max_drawdown_gbp',
    'worst_round_gbp',
    'corr_gap',
    'sanity_status',
]
comparison_df = (
    pd.DataFrame([{column: result.get(column, np.nan) for column in comparison_columns} for result in baseline_results.values()])
    .sort_values(['validation_sharpe', 'validation_corr'], ascending=False, kind='stable')
    .reset_index(drop=True)
)
search_summary_df = (
    pd.DataFrame(search_summary_rows)
    .sort_values('best_cv_score', ascending=False, kind='stable')
    .reset_index(drop=True)
)

# Select the locked winner strictly from Train CV (never pick on held-out validation!)
cv_winner_baseline = str(search_summary_df.loc[0, 'baseline'])
preferred_locked_variants = [
    f'{cv_winner_baseline} [Tuned]',
    f'{cv_winner_baseline} [Tuned Walk-Forward]',
    f'{cv_winner_baseline} [Default]',
]
best_variant_name = next(
    (v for v in preferred_locked_variants if v in baseline_results),
    str(comparison_df.loc[0, 'variant']),
)
best_validation_era_corrs = baseline_results[best_variant_name]['validation_era_corrs']
best_validation_corr = baseline_results[best_variant_name]['validation_corr']

print(f'Train-CV-Locked Winner (evaluated once on held-out validation): {best_variant_name}')
display_compact_dataframe(
    search_summary_df.drop(columns=['best_params'], errors='ignore'),
    title=f'Train CV Search Summary ({search_mode}, score_version={score_version}, refit=False):',
)
print('Full 4-layer Executive Scorecard, Money Visuals, Diagnostics & AFML Story follow in # Analysis.')
cv_analysis_artifacts = extract_cv_analysis_artifacts(
    cv_results_by_baseline,
    top_n=10,
    results_dir=results_dir,
    timestamp=timestamp,
    experiment_name=experiment_name,
    search_mode=search_mode,
    save_results=save_results,
)
del cv_results_by_baseline
gc.collect()

# Analysis

## 1. Executive Scorecard (1A–1B)
Compact **Money Table** and **Science Table** (7–8 columns each, zero horizontal truncation) plus lightweight `summary_<model_tag>.json` export for cross-notebook comparison.

In [ ]:
money_scorecard_df, science_scorecard_df = build_executive_scorecard_tables(
    comparison_df,
    best_variant_name=best_variant_name,
)
print(f'Train-CV-Locked Winner: {best_variant_name}')
display_compact_dataframe(
    money_scorecard_df,
    title='1A. Money Table (One-Wallet Staking P&L, Live-Round Return, Drawdowns & Time-Underwater):',
)
display_compact_dataframe(
    science_scorecard_df,
    title='1B. Science Table (CORR, Overfit Gap, Sharpe, Live-Round 12e CORR, PSR/DSR, Science Score % & Sanity):',
)

if save_results:
    results_dir.mkdir(parents=True, exist_ok=True)
    comparison_df.to_csv(results_dir / f'{timestamp}_{experiment_name}_comparison.csv', index=False)
    winner_summary_json_path = export_winner_scorecard_json(
        comparison_df,
        best_variant_name=best_variant_name,
        model_tag=model_tag,
        output_dir=results_dir,
        experiment_meta={
            'experiment_name': experiment_name,
            'timestamp': timestamp,
            'data_version': data_version,
            'target_col': TARGET_COL,
            'feature_set': feature_set,
            'era_stride': era_stride,
            'search_mode': search_mode,
            'score_version': score_version,
            'cv_strategy': cv_strategy,
            'eval_mode': eval_mode,
        },
    )
    print('Exported cross-notebook winner summary JSON:', winner_summary_json_path)

if export_diagnostics or save_results:
    results_dir.mkdir(parents=True, exist_ok=True)
    diagnostics_parquet_path = export_diagnostics_predictions(
        val_diag_meta,
        baseline_results[best_variant_name]['validation_preds'],
        results_dir / 'validation_predictions.parquet',
        era_col=ERA_COL,
    )
    print('Exported Numerai Diagnostics predictions Parquet (MMC & FNC):', diagnostics_parquet_path)
del money_scorecard_df, science_scorecard_df
gc.collect()

## 2. Core Performance & Money Visuals (2A–2D)
Realistic Live vs. Conservative Floor staking equity curve, 500-path block-bootstrap Monte Carlo risk, Cumulative Validation CORR across all variants, and Rolling 20-Era Validation CORR regime stability.

### NumerAPI v6.0 Dataset Discovery & True MMC vs BMC Proxy Check
Lists every published `v6.0/` file via `NumerAPI`, checks whether true `meta_model.parquet` exists, and falls back cleanly to a v6 benchmark/example predictions file labeled **BMC proxy (benchmark, NOT true MMC)**.

In [ ]:
v6_contrib_info = discover_and_fetch_v6_meta_or_benchmark(
    data_version=data_version,
    data_folder=data_folder,
    workspace_root=workspace_root,
    auto_download=True,
    verbose=True,
)

### 2A. Simulated Numerai Staking P&L Dashboard (£)

In [ ]:
from concurrent.futures import ThreadPoolExecutor as _MmcPool

val_mmc_df = val_diag_meta.assign(**{TARGET_COL: y_validation.to_numpy(dtype=np.float32)})
staking_basis_label = v6_contrib_info['basis_label']
staking_is_proxy = bool(v6_contrib_info['mmc_is_proxy'])
print(f"Active Staking Basis: {staking_basis_label}")
print(f"  -> Reason: {v6_contrib_info['reason']}")

stake_pnl_by_variant = {}
variant_era_mmc_by_variant = {}
live_scenarios_by_variant = {}
staking_summary_rows = []
_variants_list = comparison_df['variant'].tolist()

def _compute_variant_mmc_and_scenarios(variant_name):
    v_era_corrs = baseline_results[variant_name]['validation_era_corrs']
    v_era_contrib = compute_per_era_mmc(
        val_mmc_df,
        baseline_results[variant_name]['validation_preds'],
        meta_model_path=v6_contrib_info['meta_model_path'],
        benchmark_path=v6_contrib_info['benchmark_path'],
        is_benchmark_proxy=staking_is_proxy,
        era_col=ERA_COL,
        target_col=TARGET_COL,
    )
    v_scenarios = simulate_live_staking_scenarios(
        v_era_corrs,
        per_era_mmc=v_era_contrib,
        mmc_is_proxy=staking_is_proxy,
        total_capital=1000.0,
        settlement_rounds=12,
        weekly_concurrent_rounds=12,
        conservative_payout_factor=0.5,
        live_payout_factor=0.5,
        bmc_proxy_haircut=0.5,
        n_sims=500,
        block_size=12,
        random_state=random_state,
        title=variant_name,
        era_dates=val_era_dates,
    )
    return variant_name, v_era_contrib, v_scenarios

_n_mmc_workers = min(4, os.cpu_count() or 1, max(1, len(_variants_list)))
if _n_mmc_workers > 1:
    with _MmcPool(max_workers=_n_mmc_workers) as _mmc_pool:
        _mmc_outputs = list(_mmc_pool.map(_compute_variant_mmc_and_scenarios, _variants_list))
else:
    _mmc_outputs = [_compute_variant_mmc_and_scenarios(v) for v in _variants_list]

for variant_name, variant_era_contrib, _v_scenarios in _mmc_outputs:
    is_best_variant = (variant_name == best_variant_name)
    variant_slug = (
        variant_name.lower()
        .replace(' ', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('[', '')
        .replace(']', '')
        .replace('-', '_')
    )
    variant_era_corrs = baseline_results[variant_name]['validation_era_corrs']
    variant_era_mmc_by_variant[variant_name] = variant_era_contrib
    live_scenarios_by_variant[variant_name] = _v_scenarios

    title_label = f'[CV-LOCKED WINNER] {variant_name}' if is_best_variant else variant_name
    pnl_save_path = (
        results_dir / f'{timestamp}_{experiment_name}_{variant_slug}_stake_pnl_dashboard.png'
        if save_plots
        else None
    )
    haircutted_contrib = (
        pd.Series(variant_era_contrib, dtype=np.float64) * (0.5 if staking_is_proxy else 1.0)
        if variant_era_contrib is not None
        else None
    )
    if haircutted_contrib is not None:
        haircutted_contrib.name = 'bmc_proxy' if staking_is_proxy else 'mmc'
    stake_pnl_by_variant[variant_name] = plot_stake_pnl_dashboard(
        variant_era_corrs,
        per_era_mmc=haircutted_contrib,
        mmc_is_proxy=staking_is_proxy,
        conservative_pnl=live_scenarios_by_variant[variant_name]['conservative_floor']['pnl'],
        title=f'{title_label} [Realistic Live vs Conservative Floor | PF=0.5, R=12]',
        total_capital=1000.0,
        concurrent_rounds=12,
        settlement_rounds=12,
        payout_factor=0.5,
        payout_rule='auto',
        staking_mode='compound',
        era_dates=val_era_dates,
        save_path=pnl_save_path,
    )

    v_pnl = stake_pnl_by_variant[variant_name]
    c_pnl = live_scenarios_by_variant[variant_name]['conservative_floor']['pnl']
    mean_contrib = float(variant_era_contrib.mean()) if variant_era_contrib is not None and len(variant_era_contrib) > 0 else np.nan
    contrib_col_name = 'validation_bmc_proxy' if staking_is_proxy else 'validation_true_mmc'
    staking_summary_rows.append({
        'variant': variant_name,
        'staking_basis': v_pnl['score_basis'],
        'staking_mode': 'One-Wallet (Compounding, PF=0.5, R=12)',
        'validation_corr': round(float(variant_era_corrs.mean()), 4),
        contrib_col_name: round(mean_contrib, 4) if np.isfinite(mean_contrib) else np.nan,
        'start_capital': round(float(v_pnl['start_capital']), 2),
        'end_capital': round(float(v_pnl['end_capital']), 2),
        'total_return_pct': round(float(v_pnl['total_return_pct']), 2),
        'worst_dip_pct': round(float(v_pnl['money_drawdown_pct']), 2),
        'worst_dip_gbp': round(float(v_pnl['max_drawdown_gbp']), 2),
        'worst_round_gbp': round(float(v_pnl['worst_round_gbp']), 2),
    })
    print(
        f"{title_label} | Realistic Live (PF=0.5, 0.5x BMC, R=12): £{v_pnl['end_capital']:,.2f} ({v_pnl['total_return_pct']:+.2f}%, Dip {v_pnl['money_drawdown_pct']:.2f}%) | "
        f"Conservative Floor (CORR only, PF=0.5, R=12): £{c_pnl['end_capital']:,.2f} ({c_pnl['total_return_pct']:+.2f}%, Dip {c_pnl['money_drawdown_pct']:.2f}%) | "
        f"Basis: {v_pnl['score_basis']}"
    )

staking_comparison_df = pd.DataFrame(staking_summary_rows)
live_staking_scenarios_df = build_live_staking_scenarios_df(live_scenarios_by_variant)
comparison_df['staking_basis'] = comparison_df['variant'].map(
    staking_comparison_df.set_index('variant')['staking_basis']
)
if staking_is_proxy:
    comparison_df['validation_bmc_proxy'] = comparison_df['variant'].map(
        staking_comparison_df.set_index('variant')['validation_bmc_proxy']
    )
elif v6_contrib_info['basis_kind'] == 'true_mmc':
    comparison_df['validation_true_mmc'] = comparison_df['variant'].map(
        staking_comparison_df.set_index('variant')['validation_true_mmc']
    )
display_compact_dataframe(
    live_staking_scenarios_df,
    title='Dual-Scenario Realistic Live Staking Table (Conservative Floor vs. Realistic Live):',
)
best_stake_pnl = stake_pnl_by_variant[best_variant_name]
del val_mmc_df, val_diag_meta
gc.collect()
del stake_pnl_by_variant, staking_comparison_df, live_staking_scenarios_df, best_stake_pnl
gc.collect()


### 2B. Monte Carlo Block-Bootstrap Risk Analysis (£)

In [ ]:
mc_risk_by_variant = {}
explainer_printed = False
staking_is_proxy = bool(v6_contrib_info['mmc_is_proxy'])
for variant_name in comparison_df['variant'].tolist():
    is_best_variant = (variant_name == best_variant_name)
    variant_slug = (
        variant_name.lower()
        .replace(' ', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('[', '')
        .replace(']', '')
        .replace('-', '_')
    )
    mc_save_path = (
        results_dir / f'{timestamp}_{experiment_name}_{variant_slug}_monte_carlo_pnl.png'
        if save_plots
        else None
    )
    title_label = f'[CV-LOCKED WINNER] {variant_name}' if is_best_variant else variant_name
    raw_contrib = variant_era_mmc_by_variant.get(variant_name)
    haircutted_contrib = (
        pd.Series(raw_contrib, dtype=np.float64) * (0.5 if staking_is_proxy else 1.0)
        if raw_contrib is not None
        else None
    )
    if haircutted_contrib is not None:
        haircutted_contrib.name = 'bmc_proxy' if staking_is_proxy else 'mmc'
    v_mc = plot_stake_pnl_monte_carlo(
        baseline_results[variant_name]['validation_era_corrs'],
        per_era_mmc=haircutted_contrib,
        mmc_is_proxy=staking_is_proxy,
        title=f'{title_label} [Realistic Live: PF=0.5, 0.5x BMC, R=12]',
        total_capital=1000.0,
        concurrent_rounds=12,
        settlement_rounds=12,
        payout_factor=0.5,
        payout_rule='auto',
        staking_mode='compound',
        n_sims=500,
        block_size=12,
        random_state=random_state,
        era_dates=val_era_dates,
        save_path=mc_save_path,
        retain_paths=False,
    )
    mc_risk_by_variant[variant_name] = v_mc
    c_mc = live_scenarios_by_variant[variant_name]['conservative_floor']['mc']
    print('=' * 100)
    print(f"{title_label} — {v_mc['mc_verdict_text']}")
    if not explainer_printed:
        print()
        print(v_mc['mc_explainer_text'])
        explainer_printed = True
    print()
    print(v_mc['mc_summary_line'])
    print(f"  -> {v_mc['mc_drawdown_explanation_line']}")
    print(
        f"  -> Conservative Floor MC (CORR only, PF=0.5, R=12): P(Profit)={c_mc['mc_prob_profit_pct']:.1f}% | "
        f"Median=£{c_mc['mc_median_final_stake']:,.2f} | 95% Worst Dip={c_mc['mc_p95_money_drawdown_pct']:.2f}%"
    )
    print('=' * 100)
gc.collect()
del mc_risk_by_variant, v6_contrib_info, variant_era_mmc_by_variant, live_scenarios_by_variant
gc.collect()

### 2C–2D. Cumulative (2C) & Rolling 20-Era (2D) Validation CORR

In [ ]:
cum_corr_save_path = (
    results_dir / f'{timestamp}_{experiment_name}_cumulative_validation_corr.png'
    if save_plots
    else None
)
cumulative_corr_df = plot_cumulative_validation_corr(
    baseline_results,
    best_variant_name=best_variant_name,
    title='Cumulative Validation CORR Across Baseline Variants',
    save_path=cum_corr_save_path,
)

secondary_era_corrs = {
    v_name: v_res['validation_era_corrs']
    for v_name, v_res in baseline_results.items()
    if v_name != best_variant_name
}
rolling_corr_save_path = (
    results_dir / f'{timestamp}_{experiment_name}_rolling_20era_validation_corr.png'
    if save_plots
    else None
)
rolling_20era_corr = plot_rolling_validation_corr(
    best_validation_era_corrs,
    window=20,
    comparison_corrs=secondary_era_corrs,
    title=f'Rolling 20-Era Validation CORR — {best_variant_name}',
    save_path=rolling_corr_save_path,
)
del cumulative_corr_df, secondary_era_corrs, rolling_20era_corr
gc.collect()

## 3. Model Diagnostics & Residual Visuals (3A–3E)
Compact feature-neutral & error diagnostics table, peak-annotated error-direction envelope, side-by-side Top-2 variants Residuals vs. Actual plot, and model-specific interpretability.

### 3A. Model Diagnostics Table

In [ ]:
diagnostics_table_df = build_diagnostics_summary_table(
    comparison_df,
    best_variant_name=best_variant_name,
)
display_compact_dataframe(
    diagnostics_table_df,
    title='3A. Model Diagnostics Table (Feature Exposure, FNC, MAE, Forecast Bias & Hit Rate):',
)
del diagnostics_table_df
gc.collect()

### 3B. Error-Direction Envelope Analysis

In [ ]:
envelope_dfs = {}
seen_model_families = set()
envelope_variants = []
ordered_envelope_candidates = [best_variant_name] + [
    variant_name for variant_name in comparison_df['variant'].tolist() if variant_name != best_variant_name
]
for variant_name in ordered_envelope_candidates:
    if variant_name not in baseline_results:
        continue
    model_family = variant_name.split(' [')[0]
    if model_family not in seen_model_families:
        seen_model_families.add(model_family)
        envelope_variants.append(variant_name)

for variant_name in envelope_variants:
    is_best_variant = (variant_name == best_variant_name)
    variant_slug = (
        variant_name.lower()
        .replace(' ', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('[', '')
        .replace(']', '')
        .replace('-', '_')
    )
    envelope_save_path = (
        results_dir / f'{timestamp}_{experiment_name}_{variant_slug}_error_envelope.png'
        if save_plots
        else None
    )
    envelope_dfs[variant_name] = plot_era_error_envelope(
        y_validation,
        baseline_results[variant_name]['validation_preds'],
        validation_eras,
        title=f'3B. {variant_name}',
        band_size=0.15,
        is_best=is_best_variant,
        save_path=envelope_save_path,
    )

envelope_df = envelope_dfs[best_variant_name]
display_compact_dataframe(
    envelope_df.head(10),
    title=f'3B. Best Model Envelope Summary ({best_variant_name}):',
)
del envelope_dfs
gc.collect()

### 3C. Top-2 Variants Residual Diagnostics

In [ ]:
resid_save_path = (
    results_dir / f'{timestamp}_{experiment_name}_residuals_vs_actual.png'
    if save_plots
    else None
)
plot_residuals_vs_actual(
    y_validation,
    baseline_results,
    validation_eras,
    random_state=random_state,
    save_path=resid_save_path,
)

numereval_summary_df = run_numereval_diagnostics(
    y_validation,
    baseline_results[best_variant_name]['validation_preds'],
    validation_eras,
    variant_name=best_variant_name,
    plot=False,
    verbose=False,
)
strip_validation_predictions(baseline_results)
del y_validation, validation_eras
gc.collect()

### 3D–3E. Model Interpretability (3D) & Top CV Candidates (3E)

In [ ]:
display_baseline_interpretability(
    baseline_interpretability,
    results_dir=results_dir,
    timestamp=timestamp,
    experiment_name=experiment_name,
    save_results=save_plots,
)
del baseline_interpretability
gc.collect()

In [ ]:
display_top_cv_candidates(cv_analysis_artifacts)

## 4. AFML Backtest Visual Story (4A–4B)
2×2 statistical credibility dashboard for the Train-CV-Locked Winner (**PSR/DSR vs. 95%**, **Observed Eras vs. MinTRL**, **Underwater / Time-Underwater Curve**, and **HHI Concentration & Regime Health**) followed by the compact AFML reference table.

### 4A. Statistical Credibility & Regime Dashboard

In [ ]:
afml_diagnostics_df = build_afml_diagnostics_df(
    baseline_results,
    cv_trial_stats_by_baseline=cv_analysis_artifacts['trial_stats'],
    random_state=random_state,
)

winner_afml_matches = afml_diagnostics_df.loc[afml_diagnostics_df['variant'].astype(str) == str(best_variant_name)]
winner_afml_row = (
    winner_afml_matches.reset_index(drop=True).loc[0].to_dict()
    if not winner_afml_matches.empty
    else afml_diagnostics_df.reset_index(drop=True).loc[0].to_dict()
)
afml_story_save_path = (
    results_dir / f'{timestamp}_{experiment_name}_afml_backtest_story.png'
    if save_plots
    else None
)
afml_story_summary = plot_afml_backtest_story(
    best_validation_era_corrs,
    winner_afml_row,
    regime_stats=baseline_results[best_variant_name],
    title=f'4A. {best_variant_name}',
    save_path=afml_story_save_path,
)
del afml_story_summary, winner_afml_matches, winner_afml_row
gc.collect()

### 4B. AFML Backtest Credibility Reference Table

In [ ]:
compact_afml_df = build_compact_afml_reference_table(
    afml_diagnostics_df,
    best_variant_name=best_variant_name,
)
display_compact_dataframe(
    compact_afml_df,
    title='4B. AFML Backtest Credibility Reference Table (PSR, DSR, MinTRL, PBO, TuW & HHI):',
)
if save_results:
    results_dir.mkdir(parents=True, exist_ok=True)
    afml_diagnostics_df.to_csv(results_dir / f'{timestamp}_{experiment_name}_afml_diagnostics.csv', index=False)
del compact_afml_df, afml_diagnostics_df
gc.collect()

## 5. Optional Community Companion: 5A. `numereval` (Council of Elders)
Runs `councilofelders/numereval` alongside official `numerai-tools` metrics when installed locally (skips gracefully on Kaggle if unavailable). **Note:** `numereval` uses classic Spearman diagnostics and is not the official v6.0 Unity `NCORR` / `NMMC` / staking calculation.

In [ ]:
if numereval_summary_df is not None:
    print(
        f"5A. [numereval companion — {best_variant_name}] Community Spearman diagnostics "
        "(informational alongside official numerai_tools v6.0 metrics; NOT official v6.0 NCORR/NMMC):"
    )
    display_compact_dataframe(numereval_summary_df)
else:
    run_numereval_diagnostics([], [], [], variant_name=best_variant_name, plot=False, verbose=True)
del numereval_summary_df
gc.collect()

### 5B. Candidate Scoring & Fit Timing Analysis
Distribution of per-candidate CV fit & scoring times (mean, median, mode, Q1/Q3, IQR, P90/P95, failure rate, and data-driven `suggested_timeout_s`) to calibrate `max_candidate_seconds`.

In [ ]:
candidate_timing_summary_df = plot_candidate_timing_analysis(
    cv_analysis_artifacts,
    max_candidate_seconds=max_candidate_seconds,
    results_dir=results_dir,
    timestamp=timestamp,
    experiment_name=experiment_name,
    save_results=save_plots,
)
del candidate_timing_summary_df, cv_analysis_artifacts
gc.collect()